# SiPM_Characterization_Gain_Study

Studio del guadagno con il LED acceso: stessa analisi di `SiPM_Characterization_DC_Study` 
(dark count, self-trigger), da adattare ai run con il LED.

In [ ]:
## === Import principali ===
import os
import re
import contextlib, gzip, shutil, tempfile

import numpy as np
import matplotlib.pyplot as plt
import h5py
from scipy.optimize import curve_fit


# ============================================================
# LETTURA DEI FILE DELLA DAQ  (da WCFastCheck)
#
# I file hanno due formati: v1 un dataset per evento (event0, event1, ...),
# v2 un unico dataset /events/waveforms -- nato per poter leggere mentre la
# run e' in corso, che con un dataset per evento non si puo' fare. E i canali
# registrati NON sono sempre 32: si prendono dal file, non si assumono.
# ============================================================

@contextlib.contextmanager
def apri_run(path):
    """Apre un file della DAQ, compresso o no. A run finita diventa .h5.gz."""
    tmp = None
    if path.endswith(".gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False).name
        with gzip.open(path, "rb") as fin, open(tmp, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        path = tmp
    f = h5py.File(path, "r")
    try:
        yield f
    finally:
        f.close()
        if tmp:
            os.unlink(tmp)


def eventi_di(f):
    """(chiavi degli eventi, funzione che ne legge uno) per i due formati."""
    g = f["/events"]
    if "waveforms" in g:
        ds = g["waveforms"]
        return list(range(ds.shape[0])), lambda k: np.asarray(ds[k], dtype=np.float64)
    chiavi = sorted(g.keys(), key=lambda x: int(x.replace("event", "")))
    return chiavi, lambda k: np.asarray(g[k][:], dtype=np.float64)


def canali_di(f, voluti):
    """(canali nel file, indici di quelli chiesti).

    I canali si chiedono per NUMERO -- 16 e' il canale 16 del V1742 -- e non
    per posizione nella matrice: la posizione dipende da quali canali sono
    stati registrati in quella run, e cambia da una run all'altra.
    """
    lista = [int(c) for c in f["/config"].attrs["ChannelList"]]
    manca = [c for c in voluti if c not in lista]
    if manca:
        raise ValueError("Canali non nel file: %s. Il file contiene %s." % (manca, lista))
    return lista, [lista.index(c) for c in voluti]


def leggi_canale(path, canale, max_eventi=None):
    """Matrice (eventi, campioni) di un canale, in ADC int16, e SamplingTime [ns].

    Nel formato v2 il dataset si legge in un colpo solo: con eventi_di, un
    evento alla volta, mezzo milione di eventi richiedono minuti invece di
    secondi. Il v1 non lo permette e passa da eventi_di.
    """
    with apri_run(path) as f:
        sampling_time_ns = float(f["/config"].attrs["SamplingTime"]) * 1e9
        canali_file, (i_ch,) = canali_di(f, [canale])
        g = f["/events"]
        if "waveforms" in g:
            ds = g["waveforms"]
            n = ds.shape[0] if max_eventi is None else min(max_eventi, ds.shape[0])
            dati = ds[:n]
        else:
            chiavi, leggi = eventi_di(f)
            chiavi = chiavi[:max_eventi] if max_eventi is not None else chiavi
            dati = np.stack([leggi(k) for k in chiavi]).astype(np.int16)
    n_campioni = dati.shape[1] // len(canali_file)
    wf = dati.reshape(dati.shape[0], len(canali_file), n_campioni)[:, i_ch, :]
    return np.ascontiguousarray(wf), sampling_time_ns


In [ ]:
# ============================================================
# CONFIG
# ============================================================

# Il notebook sta in tools/, i dati in data/ alla radice del repo
DATA_DIR = os.path.abspath(os.path.join("..", "data/10072026/SiPM_001"))

# File del dataset. Ogni run si identifica con quello che cambia nello scan,
# letto dal nome: in uno scan in tensione la label e' la tensione (51-5V ->
# "51.5"), in uno scan in ampiezza del LED e' l'ampiezza (3LEDampl -> "3").
# Se nella lista cambiano tutte e due, la label le contiene entrambe
# ("52V_3LED"). Le chiavi per run (RUN_DA_MOSTRARE, PICCHI_MANUALI, ...) usano
# questa label; tensione e ampiezza di ogni run restano in VBIAS e LED_AMPL,
# e il fit in tensione deve prendere Vbias da VBIAS, non dalla label.
RUN_FILES = [
    "Test_SiPM_52V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_52-5V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_53V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_53-5V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_54V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_54-5V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_55V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_55-5V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_56V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    "Test_SiPM_56-5V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
    # "Test_SiPM_57V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz",
]


def parametri_da_nome(nome):
    """(tensione, ampiezza LED) dal nome del file, come stringhe; ampiezza None se manca.

    Nel nome il punto decimale e' scritto '-' (51-5V = 51.5 V), perche' un
    punto nel nome confonderebbe l'estensione.
    """
    m_v = re.search(r"_(\d+(?:-\d+)?)V_", nome)
    if m_v is None:
        raise ValueError(f"{nome}: tensione non trovata nel nome (atteso es. _51-5V_)")
    m_led = re.search(r"_(\d+(?:-\d+)?)LEDampl_", nome)
    led = m_led.group(1).replace("-", ".") if m_led else None
    return m_v.group(1).replace("-", "."), led


parametri = {nome: parametri_da_nome(nome) for nome in RUN_FILES}
tensioni = [v for v, _ in parametri.values()]
ampiezze = [a for _, a in parametri.values()]

# Lo scan si riconosce da quale parametro distingue tutti i run. Se lo fanno
# entrambi (es. un solo run) vince la tensione, come prima.
if len(set(tensioni)) == len(tensioni):
    SCAN = "HV"
elif None not in ampiezze and len(set(ampiezze)) == len(ampiezze):
    SCAN = "LED"
else:
    SCAN = "HV+LED"

# label -> file, tensione [V], ampiezza LED, descrizione per titoli e stampe
RUNS_G2 = {}
VBIAS = {}
LED_AMPL = {}
DESCRIZIONE = {}
for nome, (v, a) in parametri.items():
    if SCAN == "HV":
        label = v
    elif SCAN == "LED":
        label = a
    else:
        label = f"{v}V_{a}LED" if a is not None else f"{v}V"
    # Due run con stessa tensione e stesso LED (es. ripetuti, _0000 e _0001)
    # avrebbero la stessa label e uno sovrascriverebbe l'altro senza dirlo
    if label in RUNS_G2:
        raise ValueError(f"{nome} e {RUNS_G2[label]}: stessa tensione e stessa ampiezza LED, "
                         f"non si distinguono. Toglierne uno da RUN_FILES.")
    RUNS_G2[label] = nome
    VBIAS[label] = float(v)
    LED_AMPL[label] = float(a) if a is not None else np.nan
    DESCRIZIONE[label] = f"{v} V" + (f", LED {a}" if a is not None else "")

print(f"Scan in {SCAN}: label {list(RUNS_G2)}")

CHANNEL = 16                 # NUMERO di canale V1742, non posizione nel file
MAX_EVENTS = None            # None = tutti


# ============================================================
# LETTURA
# ============================================================

wf_gain2 = {}
SAMPLING_TIME = None
for label, nome in RUNS_G2.items():
    path = os.path.join(DATA_DIR, nome)
    wf, dt_ns = leggi_canale(path, CHANNEL, MAX_EVENTS)
    # Una run fermata prima del primo trigger produce un file valido ma vuoto:
    # senza questo controllo l'errore esce piu' avanti, oscuro (rng.choice su 0 eventi)
    if wf.shape[0] == 0:
        raise ValueError(f"{nome}: 0 eventi nel file (run senza trigger? vedere il .log)")
    if SAMPLING_TIME is not None and not np.isclose(dt_ns, SAMPLING_TIME):
        raise ValueError(f"{nome}: SamplingTime {dt_ns} ns diverso dagli altri run ({SAMPLING_TIME} ns)")
    SAMPLING_TIME = dt_ns    # [ns]
    wf_gain2[label] = wf
    print(f"[{DESCRIZIONE[label]}] {nome}: {wf.shape[0]} eventi x {wf.shape[1]} campioni, dt = {dt_ns:.4f} ns")

N_SAMPLES = next(iter(wf_gain2.values())).shape[1]


def compute_charge(wf, gate_ns, baseline_ns=None):
    """Carica per evento: somma dei campioni ADC dentro il gate [ns].

    Con baseline_ns si sottrae a ogni evento la SUA baseline, media dei
    campioni in quella finestra. Serve: la baseline si muove da un evento
    all'altro (rumore lento, sotto i 50 MHz, che il filtro non toglie), e
    sommata su tanti campioni allarga la distribuzione piu' della distanza
    fra i picchi di singolo fotoelettrone, che spariscono. Misurato sul run
    54 V senza LED: senza sottrazione nessun picco, con la sottrazione e un
    gate stretto attorno all'impulso si separano 1, 2 e 3 p.e.
    Senza baseline_ns la somma e' grezza (baseline compresa).
    I tempi ripartono da 0 sulla matrice ricevuta.
    Gli impulsi di questo SiPM sono positivi: piu' fotoni, carica piu' alta.
    """
    t = np.arange(wf.shape[1]) * SAMPLING_TIME
    sel = (t >= gate_ns[0]) & (t <= gate_ns[1])
    # float64: con le WF filtrate (float) una somma in interi troncherebbe ogni campione
    q = wf[:, sel].sum(axis=1, dtype=np.float64)
    if baseline_ns is not None:
        sel_bl = (t >= baseline_ns[0]) & (t <= baseline_ns[1])
        q -= wf[:, sel_bl].mean(axis=1, dtype=np.float64) * sel.sum()
    return q

In [ ]:
# === Run di cui si mostrano i grafici ===
# Con molti run il notebook arriva a centinaia di grafici e decine di MB: lento da aprire
# e da salvare. Le celle successive calcolano e stampano comunque per TUTTI i
# run; questa scelta salta solo il disegno.
# Per vederli tutti: commentare la seconda riga e rieseguire questa cella
# (la prima serve proprio a questo: senza, commentando la seconda il kernel
# terrebbe il valore vecchio).
RUN_DA_MOSTRARE = None          # None = grafici di tutti i run
RUN_DA_MOSTRARE = ["53", "54", "55"]      # label come in RUNS_G2: tensione ("52") o ampiezza LED ("3")

if RUN_DA_MOSTRARE is not None:
    # Una label sbagliata ("51.50" invece di "51.5") non darebbe nessun grafico senza dirlo
    sconosciute = [l for l in RUN_DA_MOSTRARE if l not in RUNS_G2]
    if sconosciute:
        raise ValueError(f"Run non nella lista: {sconosciute}. Disponibili: {list(RUNS_G2)}")


def mostra(label):
    """True se i grafici di questo run vanno disegnati."""
    return RUN_DA_MOSTRARE is None or label in RUN_DA_MOSTRARE


In [ ]:
# === Alcune waveform di ogni run, per vedere come sono i segnali ===
N_WF_SINGOLE = 6          # waveform mostrate una per pannello
N_WF_SOVRAPPOSTE = 50     # waveform sovrapposte nel pannello finale
EVENTI_SCELTI = None      # lista di indici, es. [0, 10, 250]; None = a caso
SEED = 1                  # cambiarlo per vedere altri eventi a caso

t_wf = np.arange(N_SAMPLES) * SAMPLING_TIME
eventi_per_run = {}       # gli stessi eventi si riusano nella cella del filtro

for lab in wf_gain2:
    wf_run = wf_gain2[lab]
    rng = np.random.default_rng(SEED)
    eventi = (list(EVENTI_SCELTI) if EVENTI_SCELTI is not None
              else sorted(rng.choice(len(wf_run), N_WF_SINGOLE, replace=False)))
    eventi_per_run[lab] = eventi
    if not mostra(lab):
        continue

    ncol = 3
    nrow = int(np.ceil(len(eventi) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow), sharex=True, squeeze=False)
    for ax, ev in zip(axes.flat, eventi):
        ax.plot(t_wf, wf_run[ev], lw=0.8, color="k")
        ax.set_title(f"event {ev}")
        ax.set_ylabel("ADC")
        ax.grid(True, alpha=0.3)
    for ax in axes.flat[len(eventi):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("Time [ns]")
    fig.suptitle(f"{DESCRIZIONE[lab]} — single waveforms")
    plt.tight_layout()
    plt.show()

    # Sovrapposte + media su tutti gli eventi: la media mostra dove cade il segnale
    sovr = rng.choice(len(wf_run), min(N_WF_SOVRAPPOSTE, len(wf_run)), replace=False)
    fig, ax = plt.subplots(figsize=(15, 5))
    for ev in sovr:
        ax.plot(t_wf, wf_run[ev], lw=0.5, alpha=0.3)
    ax.plot(t_wf, wf_run.mean(axis=0), lw=2, color="k", label=f"mean over {len(wf_run)} events")
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel("ADC")
    ax.set_title(f"{DESCRIZIONE[lab]} — {len(sovr)} overlaid waveforms")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# === Filtro delle waveform: grezza e filtrata a confronto ===
from scipy.signal import butter, iirnotch, sosfiltfilt, filtfilt

# Lo spettro del rumore (run 54 V senza LED) ha una riga dominante a ~98 MHz,
# larga una decina di MHz, e una minore a ~188 MHz; il segnale del SiPM sta
# sotto ~30 MHz (salita ~20 ns, discesa di centinaia di ns). Un passa-basso a
# 50 MHz toglie entrambe le righe senza toccare l'impulso. I notch sono
# l'alternativa se si vuole lasciare intatto il resto della banda: con Q alto
# pero' sono piu' stretti della riga a 98 MHz, che e' larga.
# Si filtra con filtfilt (avanti e indietro): fase nulla, quindi l'impulso non
# si sposta nel tempo e l'inizio della salita resta dov'e'.
# Lo stesso filtro si applica a tutti i run.
FILTRO_LOWPASS = True
LP_CUTOFF_MHZ = 50.0
LP_ORDINE = 4

FILTRO_NOTCH = False
NOTCH_MHZ = [97.5, 187.5]
NOTCH_Q = 10                  # Q = f0 / larghezza: 10 a 98 MHz -> ~10 MHz

USA_FILTRATE = True           # le celle successive analizzano le WF filtrate

fs_hz = 1e9 / SAMPLING_TIME   # SAMPLING_TIME e' in ns


def filtra_waveform(wf):
    """Filtra una matrice (eventi, campioni) lungo i campioni; restituisce float32."""
    out = wf.astype(np.float64)
    if FILTRO_NOTCH:
        for f0 in NOTCH_MHZ:
            b, a = iirnotch(f0 * 1e6, NOTCH_Q, fs_hz)
            out = filtfilt(b, a, out, axis=1)
    if FILTRO_LOWPASS:
        sos = butter(LP_ORDINE, LP_CUTOFF_MHZ * 1e6, btype="low", fs=fs_hz, output="sos")
        out = sosfiltfilt(sos, out, axis=1)
    return out.astype(np.float32)


wf_filt_gain2 = {label: filtra_waveform(wf_gain2[label]) for label in wf_gain2}
wf_analisi = wf_filt_gain2 if USA_FILTRATE else wf_gain2

t_f = np.arange(N_SAMPLES) * SAMPLING_TIME
filtri = " + ".join(([f"low-pass {LP_CUTOFF_MHZ:g} MHz"] if FILTRO_LOWPASS else []) +
                    ([f"notch {NOTCH_MHZ} MHz (Q={NOTCH_Q})"] if FILTRO_NOTCH else [])) or "none"
print(f"Filtro: {filtri}")

for lab in wf_gain2:
    raw, filt = wf_gain2[lab], wf_filt_gain2[lab]
    print(f"[{DESCRIZIONE[lab]}] rumore rms per evento (mediana): grezza {np.median(raw.std(axis=1)):.2f} ADC, "
          f"filtrata {np.median(filt.std(axis=1)):.2f} ADC")

    if not mostra(lab):
        continue

    # --- Stessi eventi della cella delle waveform, grezza e filtrata sovrapposte ---
    eventi = eventi_per_run[lab]
    ncol = 3
    nrow = int(np.ceil(len(eventi) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow), sharex=True, squeeze=False)
    for ax, ev in zip(axes.flat, eventi):
        ax.plot(t_f, raw[ev], lw=0.6, color="gray", alpha=0.7, label="raw")
        ax.plot(t_f, filt[ev], lw=1.2, color="red", label="filtered")
        ax.set_title(f"event {ev}")
        ax.set_ylabel("ADC")
        ax.grid(True, alpha=0.3)
    for ax in axes.flat[len(eventi):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("Time [ns]")
    axes.flat[0].legend(loc="upper right", fontsize=8)
    fig.suptitle(f"{DESCRIZIONE[lab]} — raw vs filtered ({filtri})")
    plt.tight_layout()
    plt.show()

    # --- Waveform media, prima e dopo: mostra se il filtro deforma l'impulso ---
    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.plot(t_f, raw.mean(axis=0), color="gray", label="raw mean")
    ax.plot(t_f, filt.mean(axis=0), color="red", label="filtered mean")
    ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3); ax.legend()
    ax.set_title(f"{DESCRIZIONE[lab]} — raw and filtered mean waveform")
    plt.tight_layout()
    plt.show()

In [ ]:
# === Spettro del rumore: dove sono le bande, prima e dopo il filtro ===
# Come si legge: la FFT scompone ogni waveform in sinusoidi; la potenza media
# su molti eventi dice quanta ampiezza c'e' a ogni frequenza. Il segnale del
# SiPM e' lento e sta tutto in basso (sotto ~30 MHz). Un rumore periodico
# compare invece come un picco stretto che spicca sul fondo: e' una "riga".
# Le righe si trovano da sole (find_peaks sul logaritmo della potenza grezza)
# e si marcano in tutti e due i pannelli, cosi' si vede quanto le ha tolte il filtro.
from scipy.signal import find_peaks, peak_widths

N_EVENTI_SPETTRO = 5000      # bastano per una media stabile
PROMINENZA_DECADI = 1.0      # una riga deve spiccare di 10x sul fondo vicino; con 3x
                             # si prendono anche le fluttuazioni del fondo
F_MIN_RIGHE_MHZ = 35.0       # sotto c'e' il segnale: non lo si cerca come rumore
BANDA_SEGNALE_MHZ = 30.0


def spettro_medio(wf):
    """(frequenze [MHz], potenza media [ADC^2]) su N_EVENTI_SPETTRO eventi.

    Si toglie la media di ogni evento (se no domina la frequenza zero) e si
    applica una finestra di Hann, che evita che i bordi della traccia, tagliati
    di netto, spargano potenza su tutte le frequenze.
    """
    x = wf[:N_EVENTI_SPETTRO].astype(np.float64)
    finestra = np.hanning(x.shape[1])
    x = (x - x.mean(axis=1, keepdims=True)) * finestra
    pot = (np.abs(np.fft.rfft(x, axis=1)) ** 2).mean(axis=0) / (finestra ** 2).sum()
    freq = np.fft.rfftfreq(x.shape[1], d=SAMPLING_TIME * 1e-9) / 1e6
    return freq, pot


for lab in wf_gain2:
    freq, pot_raw = spettro_medio(wf_gain2[lab])
    _, pot_filt = spettro_medio(wf_filt_gain2[lab])
    df_mhz = freq[1] - freq[0]

    # Righe di rumore sullo spettro grezzo
    cerca = freq >= F_MIN_RIGHE_MHZ
    idx, prop = find_peaks(np.log10(pot_raw[cerca]), prominence=PROMINENZA_DECADI)
    idx = np.flatnonzero(cerca)[idx]
    # larghezza a meta' altezza in scala logaritmica, convertita in MHz
    larg = peak_widths(np.log10(pot_raw), idx, rel_height=0.5)[0] * df_mhz

    print(f"=== {DESCRIZIONE[lab]} ===")
    print(f"Risoluzione in frequenza: {df_mhz:.3f} MHz  (Nyquist {freq[-1]:.0f} MHz)")
    print(f"{'riga [MHz]':>11} {'larghezza [MHz]':>16} {'sopra il fondo':>15} {'attenuazione filtro':>20}")
    for i, w, prom in zip(idx, larg, prop["prominences"]):
        att = pot_raw[i] / pot_filt[i]
        # oltre ~1e6 il filtrato e' sul pavimento numerico (float32): il numero non e' una misura
        att_txt = f"{att:19.1e}x" if att < 1e6 else f"{'> 1e6x':>20}"
        print(f"{freq[i]:11.1f} {w:16.1f} {10 ** prom:14.0f}x {att_txt}")

    if not mostra(lab):
        continue
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
    for ax, pot, titolo, colore in [(axes[0], pot_raw, "BEFORE the filter", "gray"),
                                    (axes[1], pot_filt, f"AFTER the filter ({filtri})", "red")]:
        if pot is pot_filt:
            ax.semilogy(freq, pot_raw, color="gray", lw=0.8, alpha=0.35, label="before (reference)")
        ax.semilogy(freq, pot, color=colore, lw=1, label="spectrum")
        ax.axvspan(0, BANDA_SEGNALE_MHZ, color="green", alpha=0.10, label=f"signal band (< {BANDA_SEGNALE_MHZ:g} MHz)")
        for k, (i, w) in enumerate(zip(idx, larg)):
            ax.axvspan(freq[i] - w / 2, freq[i] + w / 2, color="orange", alpha=0.35,
                       label="noise lines" if k == 0 else None)
            ax.annotate(f"{freq[i]:.1f} MHz", (freq[i], pot_raw[i]), textcoords="offset points",
                        xytext=(4, 4), fontsize=8)
        if FILTRO_LOWPASS:
            ax.axvline(LP_CUTOFF_MHZ, color="blue", ls="--", label=f"low-pass cutoff {LP_CUTOFF_MHZ:g} MHz")
        if FILTRO_NOTCH:
            for k, f0 in enumerate(NOTCH_MHZ):
                ax.axvline(f0, color="purple", ls=":", label="notch" if k == 0 else None)
        ax.set_title(titolo)
        ax.set_xlabel("Frequency [MHz]")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8, loc="upper right")
    axes[0].set_ylabel("Mean power [ADC$^2$]")
    # Sotto ~1e-4 del fondo grezzo il filtrato tocca la precisione dei float32:
    # mostrarlo schiaccerebbe tutto il resto senza aggiungere informazione
    axes[0].set_ylim(np.median(pot_raw) * 1e-4, pot_raw.max() * 10)
    fig.suptitle(f"{DESCRIZIONE[lab]} — mean spectrum over {min(N_EVENTI_SPETTRO, len(wf_gain2[lab]))} events")
    plt.tight_layout()
    plt.show()

In [ ]:
t = np.arange(N_SAMPLES) * SAMPLING_TIME

# === Inizio della salita del segnale ===
# Inizio = primo campione che supera la baseline di N_SIGMA_START volte il suo
# rumore. Si misura sia sulla waveform media sia evento per evento: la media
# dice dov'e' il segnale, la distribuzione per evento quanto si sposta (jitter),
# che e' quello che conta per scegliere il gate.
BASELINE_NS_START = (0, 180)   # regione senza segnale, prima della salita
N_SIGMA_START = 5

bl_sel = (t >= BASELINE_NS_START[0]) & (t <= BASELINE_NS_START[1])
dopo_bl = t > BASELINE_NS_START[1]
wf_media_gain2 = {}            # waveform media di ogni run, riusata qui sotto
salita_gain2 = {}              # inizio salita di ogni run

for lab in wf_analisi:
    wf_s = wf_analisi[lab].astype(np.float64)

    # Waveform media
    wf_media = wf_s.mean(axis=0)
    wf_media_gain2[lab] = wf_media
    bl_media, rms_media = wf_media[bl_sel].mean(), wf_media[bl_sel].std()
    sopra = (wf_media > bl_media + N_SIGMA_START * rms_media) & dopo_bl
    t_start_media = t[np.argmax(sopra)] if sopra.any() else np.nan
    # Tempo al 10% dell'ampiezza: meno sensibile al rumore della soglia in sigma
    picco_media = wf_media.max()
    sopra10 = (wf_media > bl_media + 0.1 * (picco_media - bl_media)) & dopo_bl
    t10_media = t[np.argmax(sopra10)]

    # Evento per evento, ciascuno con la sua baseline e il suo rumore
    bl_ev = wf_s[:, bl_sel].mean(axis=1, keepdims=True)
    rms_ev = wf_s[:, bl_sel].std(axis=1, keepdims=True)
    sopra_ev = (wf_s > bl_ev + N_SIGMA_START * rms_ev) & dopo_bl
    trovato = sopra_ev.any(axis=1)
    t_start_ev = np.where(trovato, t[np.argmax(sopra_ev, axis=1)], np.nan)
    salita_gain2[lab] = {"t_start_media": t_start_media, "t10_media": t10_media, "t_start_ev": t_start_ev}

    print(f"=== {DESCRIZIONE[lab]} ===")
    print(f"Waveform media: baseline {bl_media:.1f} ADC, rumore {rms_media:.2f} ADC")
    print(f"  inizio salita ({N_SIGMA_START} sigma) = {t_start_media:.1f} ns")
    print(f"  10% dell'ampiezza             = {t10_media:.1f} ns")
    print(f"Per evento ({trovato.sum()}/{len(trovato)} con salita trovata):")
    print(f"  mediana {np.nanmedian(t_start_ev):.1f} ns, "
          f"16-84% [{np.nanpercentile(t_start_ev, 16):.1f}, {np.nanpercentile(t_start_ev, 84):.1f}] ns, "
          f"min {np.nanmin(t_start_ev):.1f}  max {np.nanmax(t_start_ev):.1f} ns")

    if not mostra(lab):
        continue
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
    zoom = (t >= t_start_media - 100) & (t <= t_start_media + 150)
    ax1.plot(t[zoom], wf_media[zoom], color="k", label="mean waveform")
    ax1.axhline(bl_media + N_SIGMA_START * rms_media, color="gray", ls=":", label=f"baseline + {N_SIGMA_START} sigma")
    ax1.axvline(t_start_media, color="red", label=f"rise start {t_start_media:.1f} ns")
    ax1.axvline(t10_media, color="orange", ls="--", label=f"10% amplitude {t10_media:.1f} ns")
    ax1.set_xlabel("Time [ns]"); ax1.set_ylabel("ADC"); ax1.grid(True, alpha=0.3); ax1.legend()
    ax2.hist(t_start_ev[trovato], bins=np.arange(np.nanmin(t_start_ev), np.nanmax(t_start_ev) + 2 * SAMPLING_TIME, SAMPLING_TIME) - SAMPLING_TIME / 2, color="steelblue")
    ax2.set_xlabel("Rise start per event [ns]"); ax2.set_ylabel("Events"); ax2.set_yscale("log"); ax2.grid(True, alpha=0.3)
    fig.suptitle(f"{DESCRIZIONE[lab]} — rise start")
    plt.tight_layout()
    plt.show()

# === Carica per evento ===
# Gate stretto attorno all'impulso e baseline per evento subito prima: sono
# le due scelte che fanno comparire i picchi di singolo fotoelettrone (vedi
# compute_charge). Integrare tutta la traccia somma solo rumore in piu'.
#
# --- Il gate: un criterio, non un valore a occhio ---
# Il gate comincia dove la waveform media del run supera GATE_FRAZ_SALITA del
# suo massimo (sulla salita) e dura GATE_LARGHEZZA_NS. Si applica run per
# run, perche' la salita cambia un po' con la tensione.
# Perche' questa regola (misurato il 2026-10-14, cifra di merito il contrasto
# dei picchi, cella "Perche' un gate stretto"): dall'80% della salita per 40 ns
# arriva al 98, 97, 95 e 93% del contrasto del gate migliore a 53, 54, 55 e
# 56 V; il gate a occhio (220, 270) al 98, 93, 92 e 88%. L'ottimo si sposta
# con la tensione (a 53 V largo, ~216-281 ns; a 56 V stretto sul picco,
# ~227-244), e una regola legata alla forma lo segue. Scartata la regola "dove
# la media sta sopra il X% del massimo": sul lato in discesa l'impulso e' quasi
# piatto e la soglia lo incrocia in punti molto diversi (all'80%: 23 ns di gate
# a 53 V, 63 ns a 54 V).
# Dove l'impulso medio e' troppo vicino al rumore per trovarne la salita
# (picco < MIN_PICCO_SU_RUMORE_GATE volte l'rms della media nella baseline,
# lo stesso criterio di R), il gate e' quello del run affidabile piu' vicino.
GATE_FRAZ_SALITA = 0.80
GATE_LARGHEZZA_NS = 40
MIN_PICCO_SU_RUMORE_GATE = 10
BASELINE_NS = (0, 180)      # prima della salita, senza segnale

sel_bl_gate = (t >= BASELINE_NS[0]) & (t <= BASELINE_NS[1])
GATE_NS_RUN = {}            # label -> (inizio, fine) del gate [ns]
gate_info = {}
for lab in wf_media_gain2:
    m = wf_media_gain2[lab] - wf_media_gain2[lab][sel_bl_gate].mean()
    dopo = t > BASELINE_NS[1]
    i_max = np.flatnonzero(dopo)[np.argmax(m[dopo])]
    amp = m[i_max]
    sotto = np.flatnonzero(m[:i_max + 1] < GATE_FRAZ_SALITA * amp)
    i0 = sotto[-1] + 1 if len(sotto) else i_max
    gate_info[lab] = {"picco_su_rumore": amp / m[sel_bl_gate].std(), "inizio": t[i0]}
buoni_gate = [l for l in gate_info if gate_info[l]["picco_su_rumore"] >= MIN_PICCO_SU_RUMORE_GATE]
if not buoni_gate:
    raise ValueError(f"nessun run con impulso medio sopra {MIN_PICCO_SU_RUMORE_GATE} volte il rumore: "
                     f"la salita non si trova")
for lab in gate_info:
    da = lab if lab in buoni_gate else min(buoni_gate, key=lambda b: abs(VBIAS[b] - VBIAS[lab]))
    inizio = round(float(gate_info[da]["inizio"]), 1)
    GATE_NS_RUN[lab] = (inizio, round(inizio + GATE_LARGHEZZA_NS, 1))
    gate_info[lab]["da"] = da
    print(f"[{DESCRIZIONE[lab]}] gate {GATE_NS_RUN[lab][0]:.1f}-{GATE_NS_RUN[lab][1]:.1f} ns"
          + (f"  (picco/rumore {gate_info[lab]['picco_su_rumore']:.0f})" if da == lab else
             f"  dal run {DESCRIZIONE[da]}: impulso medio nel rumore (picco/rumore {gate_info[lab]['picco_su_rumore']:.0f})"))

charge_gain2 = {label: compute_charge(wf_analisi[label], GATE_NS_RUN[label], BASELINE_NS) for label in wf_analisi}

for lab in wf_analisi:
    if not mostra(lab):
        continue
    g = GATE_NS_RUN[lab]
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(t, wf_media_gain2[lab], color="k", label="mean waveform")
    ax.axvspan(*BASELINE_NS, color="green", alpha=0.15, label=f"baseline {BASELINE_NS} ns")
    ax.axvspan(*g, color="orange", alpha=0.3,
               label=f"gate {g[0]:.1f}-{g[1]:.1f} ns ({100 * GATE_FRAZ_SALITA:.0f}% of the rise + {GATE_LARGHEZZA_NS} ns)")
    ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3); ax.legend()
    ax.set_title(f"{DESCRIZIONE[lab]} — gate and baseline on the mean waveform")
    plt.tight_layout()
    plt.show()


In [ ]:
# === Distribuzione di carica di ogni run + funzioni per il fit di Poisson generalizzata ===
from math import factorial

# --- Intervallo e binnaggio ---
# L'istogramma si fa solo in CHARGE_RANGE: a parita' di numero di bin,
# un intervallo piu' stretto da' bin piu' fini. Gli eventi fuori restano
# in charge_gain2, ma non entrano ne' nell'istogramma ne' nel fit.
# Stesso intervallo per tutti i run; il numero di bin si calcola per ciascuno.
CHARGE_RANGE = None #(-1000, 2500)    # [ADC]; None = tutto l'intervallo dei dati di ogni run
N_BINS = None                   # None = regola di Freedman-Diaconis; o un numero fisso
LOG_Y = False

# Istogramma di ogni run: label -> counts, centers, bin_edges, bin_width
# (servono al fit e alla cella delle waveform per picco)
hist_gain2 = {}


# --- Funzioni per il fit (da usare quando si fitta) ---

def gen_poisson_pmf(n, mu, lam):
    """Generalized Poisson (Borel-Poisson): P(n; mu, lambda). n e' un intero scalare."""
    if n == 0:
        return np.exp(-mu)
    return mu * (mu + n * lam) ** (n - 1) * np.exp(-(mu + n * lam)) / factorial(n)


def spe_density(x, mu, lam, x0, gain, sigma0, sigma1, n_peaks):
    """Somma di gaussiane a x0 + n*gain pesate con la Poisson generalizzata."""
    y = np.zeros_like(x, dtype=float)
    for n in range(n_peaks):
        w = gen_poisson_pmf(n, mu, lam)
        sigma_n = np.sqrt(sigma0 ** 2 + n * sigma1 ** 2)
        y += w * np.exp(-(x - (x0 + n * gain)) ** 2 / (2 * sigma_n ** 2)) / (sigma_n * np.sqrt(2 * np.pi))
    return y


def fit_poisson_generalizzata(centers, counts, p0, bounds, n_peaks):
    """Fit della densita' con spe_density. p0/bounds: [mu, lambda, x0, gain, sigma0, sigma1].

    Restituisce (parametri, errori). Il fit si fa sulla densita' normalizzata,
    con errore poissoniano sui conteggi (minimo 1 per i bin vuoti).
    """
    width = centers[1] - centers[0]
    norm = counts.sum() * width
    density = counts / norm
    sigma = np.sqrt(np.maximum(counts, 1)) / norm

    def f(x, mu, lam, x0, gain, sigma0, sigma1):
        return spe_density(x, mu, lam, x0, gain, sigma0, sigma1, n_peaks)

    popt, pcov = curve_fit(f, centers, density, p0=p0, sigma=sigma,
                           bounds=bounds, maxfev=20000)
    return popt, np.sqrt(np.diag(pcov))


for lab in charge_gain2:
    charge = charge_gain2[lab]
    n_eventi = len(charge)

    if CHARGE_RANGE is None:
        lo, hi = charge.min(), charge.max()
    else:
        lo, hi = CHARGE_RANGE
    in_range = (charge >= lo) & (charge <= hi)
    charge_sel = charge[in_range]
    n_sel = len(charge_sel)
    # Se nessun evento cade nell'intervallo, np.percentile fallisce con un IndexError
    # che non dice nulla: succede quando CHARGE_RANGE e' stato scelto per un altro run
    if n_sel == 0:
        p1, p99 = np.percentile(charge, [1, 99])
        raise ValueError(f"{DESCRIZIONE[lab]}: nessun evento in CHARGE_RANGE = ({lo}, {hi}) ADC; "
                         f"le cariche stanno fra {p1:.0f} e {p99:.0f} ADC (1-99%)")

    # Freedman-Diaconis, larghezza = 2*IQR / N^(1/3), calcolata sugli eventi
    # nell'intervallo: scala con quanti eventi finiscono davvero nell'istogramma
    # e usa l'IQR invece della deviazione standard, quindi non si allarga per le code.
    if N_BINS is None:
        q25, q75 = np.percentile(charge_sel, [25, 75])
        n_bins = max(1, int(np.ceil((hi - lo) / (2 * (q75 - q25) / n_sel ** (1 / 3)))))
    else:
        n_bins = int(N_BINS)
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    counts, _ = np.histogram(charge_sel, bins=bin_edges)
    hist_gain2[lab] = {"counts": counts, "centers": centers, "bin_edges": bin_edges, "bin_width": bin_width}

    print(f"[{DESCRIZIONE[lab]}] {n_eventi} eventi")
    print(f"  carica: min {charge.min():.0f}  max {charge.max():.0f}  mediana {np.median(charge):.0f} ADC")
    print(f"  intervallo [{lo:.0f}, {hi:.0f}] ADC: {n_sel} eventi dentro, {n_eventi - n_sel} fuori")
    print(f"  binnaggio: {n_bins} bin da {bin_width:.1f} ADC")

    if not mostra(lab):
        continue
    # --- Plot della distribuzione ---
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.hist(charge_sel, bins=bin_edges, histtype="step", color="k", label=f"{DESCRIZIONE[lab]}  ({n_sel} events)")
    ax.set_xlim(lo, hi)
    ax.set_xlabel("Charge [ADC]")
    ax.set_ylabel(f"Events / {bin_width:.1f} ADC")
    ax.set_title(f"Charge distribution — {DESCRIZIONE[lab]}")
    if LOG_Y:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

# Esempio di fit su un run, da attivare quando la distribuzione e' chiara:
# h = hist_gain2["54"]
# p0     = [mu, 0.1, x0, gain, gain/4, gain/8]
# bounds = ([0, 0, x0 - gain/3, 0.7*gain, 1, 1], [30, 0.9, x0 + gain/3, 1.4*gain, gain, gain])
# popt, perr = fit_poisson_generalizzata(h["centers"], h["counts"], p0, bounds, n_peaks=10)
# ax.plot(h["centers"], spe_density(h["centers"], *popt, 10) * h["counts"].sum() * h["bin_width"], "r-")

In [ ]:
# === Fit di Poisson generalizzata: Q_pe dalle distanze fra i picchi e numero di p.e. ===
# Si fitta l'istogramma di ogni run (hist_gain2, binnaggio della cella sopra)
# con una somma di gaussiane pesate dalla Poisson generalizzata (gen_poisson_pmf).
# A differenza di spe_density, la posizione di ciascun picco visibile e' un
# parametro libero: Q_pe e' la media delle distanze fra picchi consecutivi.
# Attenzione: la media delle distanze consecutive si semplifica in
# (ultimo - primo) / (K - 1), quindi pesano solo il piedistallo e l'ultimo picco
# libero; le distanze una per una si stampano per vedere se sono costanti.
# I picchi oltre l'ultimo libero (la coda, ancora pesata dalla Poisson) stanno
# a passo costante, pari alla media delle distanze: lasciarli liberi non si
# puo', non hanno abbastanza eventi per fissarne la posizione.
#
# Numero di p.e.: mu della Poisson generalizzata = fotoelettroni primari medi
# per lampo; mu/(1-lambda) e' la media con il crosstalk compreso.
# Il numero di fotoni (mu / PDE) si calcola nella cella successiva.
# Una Poisson semplice al posto della generalizzata non serve a questo: provata
# sullo scan 52-57 V, la sua media va da 0.53 a 5.4 con chi2 ridotto fino a
# oltre 100, perche' conta come fotoni anche le celle accese dal crosstalk.
#
# Limite del modello: i picchi veri non sono gaussiani. Con bin fini (40 ADC)
# a 56.5 e 57 V hanno un fronte ripido a destra e una coda verso cariche piu'
# basse; le gaussiane simmetriche la assorbono allargando sigma0, e a 57 V il
# fit non converge piu', ne' restringendo l'intervallo ne' con bin piu' fini.
from scipy.signal import find_peaks

N_LIBERI_MAX = 8          # picchi con posizione libera, al massimo
P_CODA = 1e-4             # si aggiungono picchi finche' la probabilita' oltre e' sotto questa
# Dove i picchi non si separano (un solo massimo, es. 52 V) la ricerca automatica
# non trova il passo: lo si da' a mano, con quanti picchi lasciare liberi.
# label -> (passo iniziale [ADC], picchi liberi). Il piedistallo parte da 0.
SEMI_MANUALI = {"52": (80, 2)}
# Run da trattare come affidabili anche se il criterio sotto li scarta: a 52 V
# i picchi non si separano (sigma0 > Q_pe/2), ma Q_pe = 88 ADC sta sulla retta
# dei run a tensione piu' alta (~80 ADC estrapolati), quindi il valore si tiene.
FORZA_AFFIDABILE = {"52"}
FRAZ_PICCO_DISEGNATO = 0.01   # picchi singoli disegnati se alti almeno questa frazione del piu' alto

fit_gain2 = {}            # label -> risultati del fit, per le celle successive


def passo_autocorr(counts, bin_width):
    """Passo fra i picchi dall'autocorrelazione dell'istogramma [ADC], o nan.

    Con bin larghi quanto un quinto del passo (alle tensioni alte) find_peaks
    sui conteggi prende le fluttuazioni; l'autocorrelazione usa tutti i picchi
    insieme. Si toglie prima l'andamento lento, se no domina il lag zero.
    """
    k = max(3, len(counts) // 8) | 1
    struttura = counts - np.convolve(counts, np.ones(k) / k, mode="same")
    ac = np.correlate(struttura, struttura, mode="full")[len(counts) - 1:]
    ip, _ = find_peaks(ac)
    return ip[0] * bin_width if len(ip) else np.nan


def semina_picchi(counts, centers, passo):
    """Posizioni iniziali dei picchi visibili: massimo locale attorno a ogni posizione attesa.

    Si va avanti finche' il massimo spicca sulle valli a meta' passo; dal 2 p.e.
    in poi, se non spicca, i picchi successivi non si vedono piu'.
    """
    liscio = np.convolve(counts, np.ones(3) / 3, mode="same")
    x0, picchi = 0.0, []
    for n in range(N_LIBERI_MAX):
        vicino = np.abs(centers - (x0 + n * passo)) < passo / 3
        if not vicino.any():
            break
        i = np.flatnonzero(vicino)[np.argmax(liscio[vicino])]
        valli = [liscio[np.argmin(np.abs(centers - (x0 + (n + s) * passo)))] for s in (-0.5, 0.5)]
        if n >= 2 and liscio[i] < 1.03 * max(valli):
            break
        picchi.append(centers[i])
        if n > 0:
            x0 = centers[i] - n * passo    # la sequenza segue i picchi trovati
    return np.array(picchi)


def posizioni_picchi(m_liberi, n_tot):
    """Posizioni di tutti i picchi: i liberi, poi la coda a passo medio."""
    passo = (m_liberi[-1] - m_liberi[0]) / (len(m_liberi) - 1)
    return np.concatenate([m_liberi, m_liberi[-1] + passo * np.arange(1, n_tot - len(m_liberi) + 1)])


def picco_singolo(x, n, mu, lam, sigma0, sigma1, posizione):
    """Densita' del solo picco a n p.e.: gaussiana pesata con la Poisson generalizzata."""
    s = np.sqrt(sigma0 ** 2 + n * sigma1 ** 2)
    return gen_poisson_pmf(n, mu, lam) * np.exp(-(x - posizione) ** 2 / (2 * s ** 2)) / (s * np.sqrt(2 * np.pi))


def posa_etichette(ax, etichette, centers, counts, salti_max=8):
    """Scrive le etichette dei picchi senza sovrapporle fra loro, ai dati o alla legenda.

    Ogni etichetta e' solo il numero di p.e. (la legenda dice cosa sono), e
    parte subito sopra il suo picco. Con la scritta intera "n p.e." a 56 V
    un'etichetta era larga 1600 ADC contro 650 fra due picchi: per non
    sovrapporsi salivano di riga e restavano sospese a meta' fra due o tre
    picchi, centrate giuste ma impossibili da attribuire. Il numero da solo
    e' largo meno di un passo. Se tocca comunque un'altra etichetta, o se sotto
    di lei passa l'istogramma, sale di una riga alla volta e una linea del suo
    colore la collega al picco: chi legge non deve indovinare a quale vada.
    Dopo salti_max righe, o se esce dal grafico, si rinuncia.
    Le sovrapposizioni fra testi si misurano in pixel, a grafico gia'
    impaginato: in coordinate dei dati dipenderebbero dalla scala.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    riquadro = ax.get_window_extent(renderer)
    in_dati = ax.transData.inverted()
    occupati = [ax.get_legend().get_window_extent(renderer)] if ax.get_legend() else []
    for testo, x, y, col in etichette:
        for salto in range(salti_max + 1):
            linea = dict(arrowstyle="-", color=col, lw=0.8, shrinkA=0, shrinkB=1) if salto else None
            a = ax.annotate(testo, (x, y), textcoords="offset points", xytext=(0, 3 + 14 * salto),
                            ha="center", va="bottom", fontsize=12, fontweight="bold", color=col,
                            arrowprops=linea,
                            bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.8))
            bb = a.get_window_extent(renderer).padded(2)
            (xd0, yd0), (xd1, _) = in_dati.transform([(bb.x0, bb.y0), (bb.x1, bb.y1)])
            sotto = (centers >= xd0) & (centers <= xd1)
            libero_dai_dati = not sotto.any() or counts[sotto].max() < yd0
            dentro = riquadro.contains(bb.x0, bb.y0) and riquadro.contains(bb.x1, bb.y1)
            if dentro and libero_dai_dati and not any(bb.overlaps(o) for o in occupati):
                occupati.append(bb)
                break
            a.remove()


def densita_picchi_liberi(x, mu, lam, sigma0, sigma1, *m_liberi, n_tot):
    """Come spe_density, ma con le posizioni dei picchi date una per una."""
    m = posizioni_picchi(np.array(m_liberi), n_tot)
    return sum(picco_singolo(x, n, mu, lam, sigma0, sigma1, m[n]) for n in range(n_tot))


for lab in hist_gain2:
    h = hist_gain2[lab]
    counts, centers, w = h["counts"], h["centers"], h["bin_width"]

    if lab in SEMI_MANUALI:
        passo0, k_lib = SEMI_MANUALI[lab]
        m0 = passo0 * np.arange(k_lib, dtype=float)
    else:
        passo0 = passo_autocorr(counts, w)
        m0 = semina_picchi(counts, centers, passo0) if np.isfinite(passo0) else np.array([])
    K = len(m0)
    if K < 2:
        print(f"[{DESCRIZIONE[lab]}] meno di 2 picchi trovati: niente fit. Dare il passo in SEMI_MANUALI.")
        continue

    # mu iniziale dalla carica media; i picchi totali da quanta Poisson resta fuori
    carica_media = np.sum(counts * centers) / counts.sum()
    mu0 = max(0.2, (carica_media - m0[0]) / passo0)
    n_tot = K
    while n_tot < 60 and 1 - sum(gen_poisson_pmf(n, mu0, 0.2) for n in range(n_tot)) > P_CODA:
        n_tot += 1

    norm = counts.sum() * w
    dens = counts / norm
    err = np.sqrt(np.maximum(counts, 1)) / norm
    f = lambda x, *p: densita_picchi_liberi(x, *p, n_tot=n_tot)
    p0 = [mu0, 0.1, passo0 / 4, passo0 / 8, *m0]
    lim = ([0, 0, 1, 0, *(m0 - passo0 / 2)], [50, 0.9, 2 * passo0, 2 * passo0, *(m0 + passo0 / 2)])
    try:
        popt, pcov = curve_fit(f, centers, dens, p0=p0, sigma=err, bounds=lim, maxfev=50000)
    except RuntimeError as e:
        print(f"[{DESCRIZIONE[lab]}] fit non convergente: {e}")
        continue

    chi2_rid = np.sum(((dens - f(centers, *popt)) / err) ** 2) / (len(centers) - len(popt))
    # chi2 ridotto > 1 (il modello non descrive tutto, es. afterpulse fra i picchi):
    # gli errori di curve_fit sono sottostimati e si allargano di sqrt(chi2_rid)
    scala = np.sqrt(max(chi2_rid, 1.0))
    mu, lam, sigma0, sigma1 = popt[:4]
    e_mu = np.sqrt(pcov[0, 0]) * scala
    e_lam = np.sqrt(pcov[1, 1]) * scala
    e_lam = np.sqrt(pcov[1, 1]) * scala
    m = popt[4:]
    distanze = np.diff(m)
    q_pe = distanze.mean()
    a = np.zeros(len(popt)); a[4], a[3 + K] = -1 / (K - 1), 1 / (K - 1)
    e_qpe = np.sqrt(a @ pcov @ a) * scala
    # Se il piedistallo e' piu' largo di meta' passo i picchi non sono separati:
    # il fit restituisce il passo di partenza (succede a 57 V) e il numero non e' una misura
    affidabile = sigma0 < q_pe / 2 or lab in FORZA_AFFIDABILE
    m_tutti = posizioni_picchi(m, n_tot)

    fit_gain2[lab] = {"q_pe": q_pe, "e_q_pe": e_qpe, "mu": mu, "e_mu": e_mu, "lam": lam, "e_lam": e_lam, "e_lam": e_lam,
                      "mu_ct": mu / (1 - lam), "sigma0": sigma0, "sigma1": sigma1,
                      "posizioni": m, "distanze": distanze, "chi2_rid": chi2_rid,
                      "n_tot": n_tot, "affidabile": affidabile, "popt": popt}

    print(f"[{DESCRIZIONE[lab]}] {K} picchi liberi + coda fino a {n_tot - 1} p.e.  chi2/ndf = {chi2_rid:.2f}"
          + ("" if affidabile else "   NON AFFIDABILE: picchi non separati")
          + ("   (picchi non separati, tenuto per FORZA_AFFIDABILE)" if lab in FORZA_AFFIDABILE else ""))
    print(f"  Q_pe = {q_pe:.1f} ± {e_qpe:.1f} ADC   distanze: {np.round(distanze, 1)}")
    print(f"  mu = {mu:.3f} ± {e_mu:.3f} p.e.   lambda = {lam:.3f}   mu/(1-lambda) = {mu / (1 - lam):.3f}")
    print(f"  sigma0 = {sigma0:.1f}  sigma1 = {sigma1:.1f} ADC")

    if not mostra(lab):
        continue
    # sotto il fit i residui normalizzati (dati - fit) / errore, bin per bin:
    # devono stare attorno a 0 entro ~2 senza andamenti; un'onda dice dove il
    # modello non descrive i dati (es. le code non gaussiane dei picchi)
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(9, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    ax.step(centers, counts, where="mid", color="tab:blue", label=f"{DESCRIZIONE[lab]}")
    xx = np.linspace(centers[0], centers[-1], 2000)
    ax.plot(xx, f(xx, *popt) * norm, color="red",
            label=f"generalized Poisson: $Q_{{pe}}$ = {q_pe:.1f} ± {e_qpe:.1f} ADC, $\\mu$ = {mu:.2f}, $\\chi^2$/ndf = {chi2_rid:.2f}")

    # Ogni picco da solo, tratteggiato: peso, posizione e larghezza sono quelli del
    # fit, niente di rifittato. Mostra di cosa e' fatta la distribuzione anche dove
    # i picchi si fondono (52 V). Solo quelli abbastanza alti: la coda ne ha
    # decine invisibili, che affollerebbero il grafico.
    singoli = [picco_singolo(xx, n, mu, lam, sigma0, sigma1, m_tutti[n]) * norm for n in range(n_tot)]
    soglia = FRAZ_PICCO_DISEGNATO * max(y.max() for y in singoli)
    etichette = []
    for n, y_n in enumerate(singoli):
        if y_n.max() < soglia:
            continue
        # tab10 senza blu, rosso e grigio: sono gia' istogramma, fit e legenda
        col = plt.cm.tab10([1, 2, 4, 5, 6, 8, 9][n % 7])
        ax.plot(xx, y_n, ls="--", lw=1.2, color=col)
        # l'etichetta va sopra i DATI attorno al picco, non sopra la sola gaussiana:
        # se no finisce dentro l'istogramma e copre proprio il picco che indica
        vicino = np.abs(centers - m_tutti[n]) < max(q_pe / 4, w)
        y_lab = max(y_n.max(), counts[vicino].max() if vicino.any() else 0)
        etichette.append((f"{n}", m_tutti[n], y_lab, col))
    ax.plot([], [], ls="--", color="gray", label="single peaks (fit parameters); number = p.e.")

    ax.set_ylabel(f"Events / {w:.1f} ADC")
    ax.set_title(f"Generalized Poisson fit — {DESCRIZIONE[lab]}" + ("" if affidabile else "  (NOT RELIABLE)"))
    if LOG_Y:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.20)     # spazio per le etichette del picco piu' alto
    ax.legend(fontsize=8)
    pull = (counts - f(centers, *popt) * norm) / np.sqrt(np.maximum(counts, 1))
    axr.step(centers, pull, where="mid", color="tab:blue", lw=0.8)
    axr.axhline(0, color="red", lw=1)
    for s in (-2, 2):
        axr.axhline(s, color="gray", ls=":", lw=0.8)
    axr.set_xlabel("Charge [ADC]"); axr.set_ylabel("(data - fit) / error")
    axr.grid(True, alpha=0.3)
    plt.tight_layout()
    posa_etichette(ax, etichette, centers, counts)
    plt.show()

# --- Riepilogo su tutti i run (i grafici in funzione della tensione sono nella cella successiva) ---
if fit_gain2:
    print(f"\n{'run':>16} {'Q_pe [ADC]':>16} {'mu [p.e.]':>16} {'lambda':>8} {'mu/(1-lam)':>11}")
    for l in fit_gain2:
        r = fit_gain2[l]
        print(f"{DESCRIZIONE[l]:>16} {r['q_pe']:9.1f} ± {r['e_q_pe']:5.1f} {r['mu']:9.3f} ± {r['e_mu']:5.3f} "
              f"{r['lam']:8.3f} {r['mu_ct']:11.3f}" + ("" if r["affidabile"] else "  Q_pe non affidabile"))


In [ ]:
# === Breakdown dalla retta Q_pe(V) e numero di fotoni visti dal SiPM ===
# Il fit di Poisson generalizzata da' mu, i fotoelettroni primari medi per
# lampo: mu = N_fotoni * PDE. A LED fisso N_fotoni e' lo stesso in ogni run,
# mentre la PDE cresce con la sovratensione: per questo mu cresce con V.
# Qui si divide per la PDE del datasheet, per ottenere N_fotoni run per run.
# Se esce costante, il fit separa bene fotoni e crosstalk.
#
# Sovratensione: dV = V - V_bd, con V_bd misurato qui, dove la retta Q_pe(V)
# va a zero (Q_pe e' proporzionale a C_cella * dV). Non si usa il V_bd del
# datasheet (53 ± 5 V, a 25 °C): varia da un pezzo all'altro e con la
# temperatura, ~54 mV/°C.
#
# PDE: Hamamatsu S13360-6075PE (pitch 75 um), datasheet S13360 series
# (KAPD1052E), grafico "Overvoltage specifications ... (λ=450 nm)" a pag. 6,
# digitalizzato. Il punto a 3 V (50.4%) coincide con la tabella (50% a
# Vover = 3 V). Il grafico copre solo 2-7.9 V: sotto i 2 V la PDE viene dal
# modello PDE_max * (1 - exp(-dV / V0)) fittato su quei punti ed e' estrapolata.
# Il LED e' a 400 nm, il grafico a 450 nm: si scala per il rapporto letto dal
# grafico PDE vs lunghezza d'onda a pag. 4 (misurato a Vover = 3 V).
# Ipotesi: lo stesso rapporto vale a ogni sovratensione. Non e' verificato.
#
# Gli errori su N_fotoni vengono solo da mu. I sistematici non sono inclusi:
# PDE "tipica" del datasheet, digitalizzazione (~1%), rapporto 400/450 nm,
# estrapolazione sotto 2 V.

# --- Q_pe dell'impulso intero ---
# Il gate stretto (GATE_NS_RUN) raccoglie solo una parte dell'impulso, che scende
# in centinaia di ns: il Q_pe del fit e' la carica di un p.e. DENTRO IL GATE.
# Per ogni run R = carica su tutta la finestra del segnale / carica nel gate,
# sugli stessi eventi e con la stessa baseline per evento; Q_pe_tot = R * Q_pe.
# R e' il rapporto delle cariche MEDIE, cioe' degli integrali della waveform
# media; l'errore e' quello statistico, dalla dispersione evento per evento.
# Vale per ogni numero di p.e. se la forma dell'impulso non cambia con
# l'ampiezza: le medie per picco dell'ultima cella lo fanno pensare, non e'
# misurato.
# Per il light yield: Q_pe_tot va con cariche integrate su TUTTO l'impulso,
# Q_pe con cariche integrate nello STESSO gate. Mescolarli sbaglia di un
# fattore ~4.
# Misurato: R 4.04-4.24 da 53 V in su, cresce con la tensione. La forma
# normalizzata e' la stessa, salvo una coda un po' piu' alta a tensione alta:
# probabilmente afterpulse e crosstalk ritardato, non verificato. Questi non
# fanno parte del guadagno del datasheet: Q_pe_tot li contiene.
# A bassa tensione R NON si misura: nella waveform media resta un'oscillazione
# sincrona col trigger, ~1 ADC con periodo ~35 ns, e a 52 V l'impulso medio e'
# alto 2.6 ADC. Integrata su 1100 ns, l'oscillazione pesa quanto il segnale:
# R = 2.58 a 52 V, che cambia del 15% spostando la fine della finestra. Dove il
# picco dell'impulso medio non supera MIN_PICCO_SU_RUMORE volte l'rms della
# waveform media nella baseline (che contiene quell'oscillazione), si usa R
# del run affidabile piu' vicino in tensione, e il punto e' segnato a parte.
# La soglia 10 e' una scelta: lascia fuori 52 V (3) e 52.5 V (6, R = 3.76
# contro 4.04 a 53 V), tiene 53 V (11), il cui R coincide con quello a 53.5 V.
# L'errore su R e' solo statistico: quello dell'oscillazione non e' stimato.
# --- Fine della finestra lunga: dove la carica smette di crescere ---
# Criterio: la finestra va dalla fine della baseline al primo istante in cui
# la carica cumulata della waveform media raggiunge FRAZ_CARICA_FINESTRA del
# suo valore finale (media della cumulata negli ultimi 300 ns della traccia),
# preso il piu' tardo fra i run e arrotondato per eccesso a 50 ns. Solo i run
# con impulso medio alto almeno MIN_PICCO_SU_RUMORE_FINESTRA volte il rumore:
# sotto, la cumulata oscilla di qualche % e il 99% si incontra a caso.
# Misurato il 2026-10-14: a 55 V la cumulata e' al 98.4% a 800 ns, al 99.6% a
# 900 ns, e oltre oscilla di +-2% (rumore, non segnale). Integrare oltre non
# aggiunge carica ma solo rumore; e nelle run dei pannelli con PostTrigger 40
# la traccia finisce ~900 ns dopo la salita, quindi una finestra fino a fine
# traccia non sarebbe nemmeno riproducibile.
# Prova che la sottrazione della baseline e' giusta: la cumulata SATURA (vedi il
# grafico qui sotto). Se continuasse a crescere o calasse, la baseline sarebbe
# sbagliata.
FRAZ_CARICA_FINESTRA = 0.99
MIN_PICCO_SU_RUMORE_FINESTRA = 30
CODA_PLATEAU_NS = 300

i_bl_fine = np.searchsorted(t, BASELINE_NS[1])
cumulate = {}
for l in gate_info:
    if gate_info[l]["picco_su_rumore"] < MIN_PICCO_SU_RUMORE_FINESTRA:
        continue
    m = wf_media_gain2[l] - wf_media_gain2[l][sel_bl_gate].mean()
    cum = np.cumsum(m[i_bl_fine:])
    plateau = cum[t[i_bl_fine:] >= t[-1] - CODA_PLATEAU_NS].mean()
    t_frac = t[i_bl_fine:][np.argmax(cum >= FRAZ_CARICA_FINESTRA * plateau)]
    cumulate[l] = {"cum": cum / plateau, "t_frac": t_frac}
if not cumulate:
    raise ValueError(f"nessun run con impulso medio sopra {MIN_PICCO_SU_RUMORE_FINESTRA} volte il rumore: "
                     f"la fine della finestra non si misura")
FINE_SEGNALE_NS = float(np.ceil(max(c["t_frac"] for c in cumulate.values()) / 50) * 50)
FINESTRA_SEGNALE_NS = (BASELINE_NS[1], FINE_SEGNALE_NS)
print(f"Fine della finestra lunga: carica cumulata al {100 * FRAZ_CARICA_FINESTRA:.0f}% del valore finale a "
      + ", ".join(f"{DESCRIZIONE[l]} {c['t_frac']:.0f} ns" for l, c in cumulate.items())
      + f"  ->  finestra {FINESTRA_SEGNALE_NS[0]:.0f}-{FINESTRA_SEGNALE_NS[1]:.0f} ns")

fig, ax = plt.subplots(figsize=(12, 4.5))
for l, c in cumulate.items():
    ax.plot(t[i_bl_fine:], c["cum"], lw=1, label=DESCRIZIONE[l])
ax.axhline(1, color="gray", lw=0.8)
ax.axhline(FRAZ_CARICA_FINESTRA, color="gray", ls=":", label=f"{100 * FRAZ_CARICA_FINESTRA:.0f}%")
ax.axvline(FINE_SEGNALE_NS, color="red", ls="--", label=f"end of the full window: {FINE_SEGNALE_NS:.0f} ns")
ax.set_ylim(0.8, 1.05)
ax.set_xlabel("End of the integration window [ns]"); ax.set_ylabel("Cumulative charge / final value")
ax.set_title("Mean pulse: cumulative charge from the end of the baseline (it must saturate)")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

MIN_PICCO_SU_RUMORE = 10

sel_bl_m = (t >= BASELINE_NS[0]) & (t <= BASELINE_NS[1])
for l in fit_gain2:
    r = fit_gain2[l]
    q_gate = charge_gain2[l]
    q_tutto = compute_charge(wf_analisi[l], FINESTRA_SEGNALE_NS, BASELINE_NS)
    m_t, m_g = q_tutto.mean(), q_gate.mean()
    cv = np.cov(q_tutto, q_gate) / len(q_gate)
    R = m_t / m_g
    e_R = abs(R) * np.sqrt(cv[0, 0] / m_t ** 2 + cv[1, 1] / m_g ** 2 - 2 * cv[0, 1] / (m_t * m_g))
    w_m = wf_media_gain2[l] - wf_media_gain2[l][sel_bl_m].mean()
    picco_su_rumore = w_m.max() / w_m[sel_bl_m].std()
    r.update({"R_gate": R, "e_R_gate": e_R, "R_misurato": R, "e_R_misurato": e_R,
              "picco_su_rumore": picco_su_rumore, "R_affidabile": picco_su_rumore >= MIN_PICCO_SU_RUMORE})

R_buoni = [l for l in fit_gain2 if fit_gain2[l]["R_affidabile"]]
if not R_buoni:
    raise ValueError(f"nessun run con impulso medio sopra {MIN_PICCO_SU_RUMORE} volte il rumore: R non misurabile")
for l in fit_gain2:
    r = fit_gain2[l]
    if not r["R_affidabile"]:
        vicino = min(R_buoni, key=lambda b: abs(VBIAS[b] - VBIAS[l]))
        r["R_gate"], r["e_R_gate"], r["R_da"] = fit_gain2[vicino]["R_gate"], fit_gain2[vicino]["e_R_gate"], vicino
    r["q_pe_tot"] = r["R_gate"] * r["q_pe"]
    r["e_q_pe_tot"] = r["q_pe_tot"] * np.hypot(r["e_q_pe"] / r["q_pe"], r["e_R_gate"] / r["R_gate"])

print(f"Q_pe dell'impulso intero: finestra {FINESTRA_SEGNALE_NS[0]:.0f}-{FINESTRA_SEGNALE_NS[1]:.0f} ns, "
      f"gate di ogni run dal {100 * GATE_FRAZ_SALITA:.0f}% della salita per {GATE_LARGHEZZA_NS} ns")
print(f"{'run':>16} {'picco/rumore':>12} {'R misurato':>16} {'R usato':>8} {'Q_pe gate [ADC]':>16} {'Q_pe tot [ADC]':>17}")
for l in fit_gain2:
    r = fit_gain2[l]
    print(f"{DESCRIZIONE[l]:>16} {r['picco_su_rumore']:12.0f} {r['R_misurato']:8.3f} ± {r['e_R_misurato']:5.3f} "
          f"{r['R_gate']:8.3f} {r['q_pe']:9.1f} ± {r['e_q_pe']:5.1f} {r['q_pe_tot']:10.1f} ± {r['e_q_pe_tot']:5.1f}"
          + ("" if r["R_affidabile"] else f"  R dal run {DESCRIZIONE[r['R_da']]}: impulso medio nel rumore")
          + ("" if r["affidabile"] else "  Q_pe non affidabile"))

# --- Guadagno: da Q_pe_tot [ADC x campione] a elettroni per fotoelettrone ---
# G = Q_pe_tot * (V per conteggio) * dt / (R_ingresso * guadagno scheda) / e
#  - V1742: 1 Vpp su 12 bit, quindi 1/4096 V per conteggio;
#  - dt dal file (SAMPLING_TIME), perche' la carica e' una SOMMA di campioni;
#  - la corrente del SiPM entra nella scheda di amplificazione (x16, alimentata
#    a 12 V) attraverso la sua impedenza d'ingresso, assunta 50 ohm: NON
#    verificato, se e' diversa il guadagno scala con lei.
ADC_VPP = 1.0                 # [V] dinamica del V1742
ADC_BIT = 12
R_INGRESSO_OHM = 50.0         # impedenza d'ingresso della scheda di amplificazione (assunta)
GUADAGNO_AMPLI = 16.0         # [V/V] scheda di amplificazione
E_CARICA = 1.602176634e-19    # [C]
# Riferimento: datasheet S13360-6075PE, guadagno 4.0e6 alla tensione operativa V_bd + 3 V
G_DATASHEET, DV_DATASHEET = 4.0e6, 3.0

ELETTRONI_PER_ADC = (ADC_VPP / 2 ** ADC_BIT) * (SAMPLING_TIME * 1e-9) / (R_INGRESSO_OHM * GUADAGNO_AMPLI) / E_CARICA
print(f"\n1 ADC x campione = {ELETTRONI_PER_ADC:.0f} elettroni (dt = {SAMPLING_TIME:.3f} ns, scheda x{GUADAGNO_AMPLI:g}, "
      f"{R_INGRESSO_OHM:g} ohm)")

PDE_DV_V = np.array([2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 7.85])
PDE_450_PERC = np.array([41.0, 46.9, 50.4, 53.4, 55.8, 59.1, 61.3, 62.7, 63.9, 64.7, 65.3, 65.6, 65.8])
RAPPORTO_400_450 = 45.4 / 50.0     # PDE(400 nm) / PDE(450 nm), pag. 4 del datasheet
LAMBDA_LED_NM = 400


def modello_pde(dv, pde_max, v0):
    """PDE in funzione della sovratensione: sale da zero e satura."""
    return pde_max * (1 - np.exp(-dv / v0))


p_pde, _ = curve_fit(modello_pde, PDE_DV_V, PDE_450_PERC, p0=[66, 2])
res_pde = np.abs(modello_pde(PDE_DV_V, *p_pde) - PDE_450_PERC).max()
print(f"PDE datasheet (450 nm): PDE_max = {p_pde[0]:.1f}%, V0 = {p_pde[1]:.2f} V, "
      f"scarto massimo dai punti {res_pde:.1f}%")


def pde_led(dv):
    """PDE [frazione] alla lunghezza d'onda del LED per sovratensione dv [V]."""
    return modello_pde(dv, *p_pde) / 100 * RAPPORTO_400_450


# --- Breakdown: retta Q_pe = k * (V - V_bd), pesata con gli errori, sui run affidabili ---
labs_v = [l for l in fit_gain2 if fit_gain2[l]["affidabile"]]
v_ok = np.array([VBIAS[l] for l in labs_v])
q_ok = np.array([fit_gain2[l]["q_pe"] for l in labs_v])
eq_ok = np.array([fit_gain2[l]["e_q_pe"] for l in labs_v])
(k_q, b_q), cov_q = np.polyfit(v_ok, q_ok, 1, w=1 / eq_ok, cov="unscaled")
chi2_q = np.sum(((q_ok - (k_q * v_ok + b_q)) / eq_ok) ** 2) / (len(v_ok) - 2)
# errori allargati di sqrt(chi2): gli scarti dalla retta sono piu' grandi degli errori del fit
cov_q = cov_q * max(chi2_q, 1.0)
V_BD = -b_q / k_q
# propagazione su V_bd = -b/k
grad = np.array([b_q / k_q ** 2, -1 / k_q])
E_V_BD = np.sqrt(grad @ cov_q @ grad)
print(f"Q_pe = {k_q:.1f} ADC/V * (V - V_bd),  V_bd = {V_BD:.3f} ± {E_V_BD:.3f} V,  chi2/ndf = {chi2_q:.2f}")
# --- Retta di Q_pe_tot: capacita' della cella e, per confronto, il suo V_bd ---
# V_bd resta quello del gate (sopra): fotoni e sovratensioni non cambiano.
# Se R fosse costante le due rette darebbero lo stesso V_bd; R cresce un po'
# con la tensione e lo sposta: la differenza dice quanto pesa.
q_tot_ok = np.array([fit_gain2[l]["q_pe_tot"] for l in labs_v])
eq_tot_ok = np.array([fit_gain2[l]["e_q_pe_tot"] for l in labs_v])
(k_t, b_t), cov_t = np.polyfit(v_ok, q_tot_ok, 1, w=1 / eq_tot_ok, cov="unscaled")
chi2_t = np.sum(((q_tot_ok - (k_t * v_ok + b_t)) / eq_tot_ok) ** 2) / (len(v_ok) - 2)
cov_t = cov_t * max(chi2_t, 1.0)
V_BD_TOT = -b_t / k_t
grad_t = np.array([b_t / k_t ** 2, -1 / k_t])
E_V_BD_TOT = np.sqrt(grad_t @ cov_t @ grad_t)
# G = C_cella * (V - V_bd) / e: la pendenza da' la capacita' della cella
k_g = k_t * ELETTRONI_PER_ADC
print(f"Q_pe_tot = {k_t:.1f} ADC/V * (V - V_bd),  V_bd = {V_BD_TOT:.3f} ± {E_V_BD_TOT:.3f} V "
      f"(dal gate {V_BD:.3f}),  chi2/ndf = {chi2_t:.2f}")
print(f"G = {k_g:.3g} /V * (V - V_bd),  C_cella = {k_g * E_CARICA * 1e15:.0f} fF;  "
      f"a V_bd + {DV_DATASHEET:g} V: G = {k_g * DV_DATASHEET:.2e} (datasheet {G_DATASHEET:.1e})")

# --- Numero di fotoni per run ---
fotoni_gain2 = {}
print(f"\n{'run':>16} {'dV [V]':>7} {'PDE':>7} {'mu [p.e.]':>16} {'N fotoni':>16}")
for l in fit_gain2:
    if SCAN == "LED":
        break    # in uno scan in ampiezza LED la tensione e' fissa: niente da confrontare
    dv = VBIAS[l] - V_BD
    pde = pde_led(dv) if dv > 0 else np.nan
    r = fit_gain2[l]
    estrapolata = dv < PDE_DV_V[0]
    fotoni_gain2[l] = {"dv": dv, "pde": pde, "n": r["mu"] / pde, "e_n": r["e_mu"] / pde,
                       "estrapolata": estrapolata}
    print(f"{DESCRIZIONE[l]:>16} {dv:7.2f} {pde * 100:6.1f}% {r['mu']:9.3f} ± {r['e_mu']:5.3f} "
          f"{r['mu'] / pde:9.2f} ± {r['e_mu'] / pde:5.2f}"
          + ("  PDE estrapolata (dV < 2 V)" if estrapolata else "")
          + ("" if r["affidabile"] else "  Q_pe non affidabile"))

if fotoni_gain2:
    # media pesata solo dove la PDE e' letta dal datasheet e Q_pe e' affidabile
    buoni = [l for l in fotoni_gain2 if not fotoni_gain2[l]["estrapolata"] and fit_gain2[l]["affidabile"]]
    n_b = np.array([fotoni_gain2[l]["n"] for l in buoni])
    w_b = 1 / np.array([fotoni_gain2[l]["e_n"] for l in buoni]) ** 2
    N_FOTONI = np.sum(w_b * n_b) / w_b.sum()
    chi2_n = np.sum(w_b * (n_b - N_FOTONI) ** 2) / (len(n_b) - 1)
    print(f"\nN fotoni (media pesata, {len(buoni)} run con PDE dal datasheet): {N_FOTONI:.2f}, "
          f"dispersione fra i run (std) {n_b.std(ddof=1):.2f}, chi2/ndf rispetto alla media {chi2_n:.2f}")

# --- Grafici ---
# Sotto ogni fit i residui normalizzati (dati - fit) / errore, sui punti usati nel fit.
fig, assi = plt.subplots(2, 3, figsize=(21, 8), sharex="col", gridspec_kw={"height_ratios": [3, 1]})
(ax1, ax3, ax2), (r1, r3, r2) = assi


def residui(ax_r, x, y, e, modello, colore="k"):
    ax_r.errorbar(x, (y - modello) / e, np.ones_like(x), fmt="o", color=colore, ms=4)
    ax_r.axhline(0, color="red", lw=1)
    for s in (-2, 2):
        ax_r.axhline(s, color="gray", ls=":", lw=0.8)
    ax_r.set_xlabel("Bias voltage [V]"); ax_r.set_ylabel("(data - fit) / error")
    ax_r.grid(True, alpha=0.3)


labs = list(fit_gain2)
x_all = np.array([VBIAS[l] for l in labs])
q_all = np.array([fit_gain2[l]["q_pe"] for l in labs])
eq_all = np.array([fit_gain2[l]["e_q_pe"] for l in labs])
ok = np.array([fit_gain2[l]["affidabile"] for l in labs])
ax1.errorbar(x_all[ok], q_all[ok], eq_all[ok], fmt="o", color="k", label="generalized Poisson fit")
if (~ok).any():
    ax1.errorbar(x_all[~ok], q_all[~ok], eq_all[~ok], fmt="x", color="gray", label="not reliable")
xr = np.linspace(V_BD, x_all.max() + 0.2, 100)
ax1.plot(xr, k_q * xr + b_q, color="red",
         label=f"linear fit: {k_q:.1f} ADC/V, $V_{{bd}}$ = {V_BD:.2f} ± {E_V_BD:.2f} V, $\\chi^2$/ndf = {chi2_q:.2f}")
ax1.axvline(V_BD, color="red", ls=":", lw=1)
ax1.set_ylabel("$Q_{pe}$ [ADC]")
ax1.set_title(f"Charge per photoelectron (gate: {100 * GATE_FRAZ_SALITA:.0f}% of the rise + {GATE_LARGHEZZA_NS} ns)")
residui(r1, v_ok, q_ok, eq_ok, k_q * v_ok + b_q)
ax1.grid(True, alpha=0.3); ax1.legend(fontsize=8)

# --- Q_pe dell'impulso intero, e il guadagno sull'asse a destra (fattore costante) ---
q_tot_all = np.array([fit_gain2[l]["q_pe_tot"] for l in labs])
eq_tot_all = np.array([fit_gain2[l]["e_q_pe_tot"] for l in labs])
r_mis = np.array([fit_gain2[l]["R_affidabile"] for l in labs])
for sel, kw in [(ok & r_mis, dict(fmt="o", color="k", label="$Q_{pe}$ (fit) x R of the run")),
                (ok & ~r_mis, dict(fmt="o", mfc="white", color="k",
                                   label="R from the nearest run (mean pulse in the noise)")),
                (~ok, dict(fmt="x", color="gray", label="not reliable"))]:
    if sel.any():
        ax3.errorbar(x_all[sel], q_tot_all[sel], eq_tot_all[sel], **kw)
ax3.plot(xr, k_t * xr + b_t, color="red",
         label=f"linear fit: $C_{{cell}}$ = {k_g * E_CARICA * 1e15:.0f} fF, "
               f"$V_{{bd}}$ = {V_BD_TOT:.2f} ± {E_V_BD_TOT:.2f} V, $\\chi^2$/ndf = {chi2_t:.2f}")
ax3.axvline(V_BD, color="red", ls=":", lw=1)
ax3.set_ylabel("$Q_{pe}$ full pulse [ADC]")
ax3.set_title(f"Full-pulse charge per p.e. ({FINESTRA_SEGNALE_NS[0]:.0f}-{FINESTRA_SEGNALE_NS[1]:.0f} ns) and gain")
sec3 = ax3.secondary_yaxis("right", functions=(lambda q: q * ELETTRONI_PER_ADC / 1e6,
                                               lambda g: g * 1e6 / ELETTRONI_PER_ADC))
sec3.set_ylabel(f"Gain [$10^6$]  (amplifier x{GUADAGNO_AMPLI:g}, {R_INGRESSO_OHM:g} $\\Omega$)")
ax3.grid(True, alpha=0.3); ax3.legend(fontsize=8)
residui(r3, v_ok, q_tot_ok, eq_tot_ok, k_t * v_ok + b_t)

if fotoni_gain2:
    lf = list(fotoni_gain2)
    xf = np.array([VBIAS[l] for l in lf])
    nf = np.array([fotoni_gain2[l]["n"] for l in lf])
    enf = np.array([fotoni_gain2[l]["e_n"] for l in lf])
    est = np.array([fotoni_gain2[l]["estrapolata"] for l in lf])
    # mu viene dallo stesso fit di Q_pe: dove quello non e' affidabile (57 V) non lo e' neanche mu
    aff = np.array([fit_gain2[l]["affidabile"] for l in lf])
    for sel, kw in [(~est & aff, dict(fmt="o", color="k", label="PDE from datasheet")),
                    (est & aff, dict(fmt="o", mfc="white", color="k",
                                     label="PDE extrapolated ($\\Delta V$ < 2 V)")),
                    (~aff, dict(fmt="x", color="gray", label="fit not reliable"))]:
        if sel.any():
            ax2.errorbar(xf[sel], nf[sel], enf[sel], **kw)
    ax2.axhline(N_FOTONI, color="red", ls="--", label=f"weighted mean = {N_FOTONI:.2f}, $\\chi^2$/ndf = {chi2_n:.2f}")
    e_b = np.array([fotoni_gain2[l]["e_n"] for l in buoni])
    residui(r2, np.array([VBIAS[l] for l in buoni]), n_b, e_b, N_FOTONI)
    ax2.set_ylim(0, None)
    # sovratensione sull'asse in alto: e' la variabile della PDE
    sec = ax2.secondary_xaxis("top", functions=(lambda v: v - V_BD, lambda d: d + V_BD))
    sec.set_xlabel("Overvoltage [V]")
ax2.set_ylabel("Number of photons")
ax2.set_title(f"Photons on the SiPM = $\\mu$ / PDE ({LAMBDA_LED_NM} nm)", pad=28)
ax2.grid(True, alpha=0.3); ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# === Perche' un gate stretto: confronto con il gate intero e scan della larghezza ===
# Dimostrazione della scelta del gate (GATE_NS_RUN) su poche tensioni (TENSIONI_CONFRONTO):
# non cambia nulla delle celle precedenti.
#
# Cifra di merito: il CONTRASTO dei picchi, |<exp(2 pi i q / G)>| sugli eventi,
# cioe' quanto la distribuzione di carica e' periodica con passo G (1 p.e.).
# Intuitivamente: ogni evento e' una lancetta su un orologio la cui circonferenza
# vale 1 p.e. Gli eventi nei picchi (0, 1, 2... p.e.) puntano tutti nello stesso
# verso, quelli fuori in versi a caso; il contrasto e' la lunghezza della
# lancetta media. Lo stesso si vede nell'istogramma "ripiegato" ogni G (pannello
# centrale): un picco se la periodicita' c'e', una riga piatta se no.
# Vale 1 se tutti gli eventi stanno in picchi infinitamente stretti, ~0 se di
# periodicita' non ce n'e' (il fondo statistico con 1e5 eventi e' ~0.003).
# Per picchi gaussiani larghi sigma vale exp(-2 pi^2 sigma^2 / G^2).
# Perche' non sigma0 da un fit: allargando il gate i picchi restano, ma sopra
# un fondo continuo (impulsi in piu' dentro il gate: afterpulse, buio). Un
# modello con una sola larghezza non lo descrive, e il fit salta fra una
# soluzione "picchi stretti" e una "gobba unica": a 55 V G/sigma0 passava da
# 2.5 a 1.2 fra 100 e 110 ns di gate, con gli istogrammi quasi uguali. Il
# contrasto non ha modello, e il fondo continuo semplicemente non vi contribuisce.
#
# G di un gate qualsiasi: Q_pe del gate stretto (fit libero) scalato per il
# rapporto fra gli integrali della waveform media nei due gate. Verificato
# lasciando G libero nel fit a 53 e 55 V: stesso valore entro il 3%. Il
# contrasto si prende comunque al massimo per G entro +-10%, cosi' un errore
# su G non lo abbassa.
#
# Baseline: la stessa per tutti i gate, quella che finisce prima del gate piu'
# anticipato (il gate intero). Cosi' l'unica differenza fra i gate e' il gate.
# Per questo la carica del gate stretto qui differisce di poco da charge_gain2.
TENSIONI_CONFRONTO = ["53", "55"]  # label come in RUNS_G2
PASSO_SCAN_NS = 10                 # passo sulla fine del gate nello scan
MARGINE_INIZIO_NS = 20
FRAZIONE_CODA = 0.10

sconosciute = [l for l in TENSIONI_CONFRONTO if l not in wf_analisi]
if sconosciute:
    raise ValueError(f"TENSIONI_CONFRONTO: {sconosciute} non fra i run. Disponibili: {list(wf_analisi)}")


def q_pe_gate_stretto(q, lab):
    """Q_pe dal fit libero sul gate stretto, come nella cella del fit."""
    lo, hi = np.percentile(q, (0.1, 99.5))
    q = q[(q >= lo) & (q <= hi)]
    q25, q75 = np.percentile(q, [25, 75])
    edges = np.linspace(lo, hi, max(1, int(np.ceil((hi - lo) / (2 * (q75 - q25) / len(q) ** (1 / 3))))) + 1)
    counts, _ = np.histogram(q, edges)
    centers, w = 0.5 * (edges[1:] + edges[:-1]), edges[1] - edges[0]
    if lab in SEMI_MANUALI:
        passo0, k_lib = SEMI_MANUALI[lab]
        m0 = passo0 * np.arange(k_lib, dtype=float)
    else:
        passo0 = passo_autocorr(counts, w)
        m0 = semina_picchi(counts, centers, passo0)
    K = len(m0)
    mu0 = max(0.2, (np.sum(counts * centers) / counts.sum() - m0[0]) / passo0)
    n_tot = K
    while n_tot < 60 and 1 - sum(gen_poisson_pmf(n, mu0, 0.2) for n in range(n_tot)) > P_CODA:
        n_tot += 1
    norm = counts.sum() * w
    f = lambda x, *p: densita_picchi_liberi(x, *p, n_tot=n_tot)
    lim = ([0, 0, 1, 0, *(m0 - passo0 / 2)], [50, 0.9, 2 * passo0, 2 * passo0, *(m0 + passo0 / 2)])
    popt, _ = curve_fit(f, centers, counts / norm, p0=[mu0, 0.1, passo0 / 4, passo0 / 8, *m0],
                        sigma=np.sqrt(np.maximum(counts, 1)) / norm, bounds=lim, maxfev=50000)
    m = popt[4:]
    return (m[-1] - m[0]) / (K - 1)


def contrasto_picchi(q, g):
    """(contrasto, passo usato, posizione del piedistallo) massimizzando sul passo entro +-10%.

    La fase della media complessa dice dove cade il pettine dei picchi: serve
    a riportare il piedistallo a 0 nell'istogramma in unita' di p.e.
    """
    migliore = (0.0, g, 0.0)
    for gp in g * np.linspace(0.9, 1.1, 41):
        z = np.mean(np.exp(2j * np.pi * q / gp))
        if abs(z) > migliore[0]:
            migliore = (abs(z), gp, np.angle(z) / (2 * np.pi) * gp)
    return migliore


confronto_gate = {}
for lab in TENSIONI_CONFRONTO:
    wf_l = wf_analisi[lab]
    media = wf_media_gain2[lab]
    gate_s = GATE_NS_RUN[lab]

    # --- Gate intero: dalla salita alla fine della coda della waveform media ---
    # Ai bordi della traccia il filtro (filtfilt) lascia oscillazioni sopra il 10%:
    # l'inizio si cerca dal massimo all'indietro, la fine esclude gli ultimi 20 ns.
    bl_m = media[(t >= BASELINE_NS[0]) & (t <= BASELINE_NS[1])].mean()
    sopra = media - bl_m > FRAZIONE_CODA * (media.max() - bl_m)
    i_max = int(np.argmax(media))
    sotto_prima = np.flatnonzero(~sopra[:i_max])
    i_salita = sotto_prima[-1] + 1 if len(sotto_prima) else 0
    i_fine = np.flatnonzero(sopra & (t <= t[-1] - 20))[-1]
    gate_intero = (t[i_salita] - MARGINE_INIZIO_NS, t[i_fine])

    # --- Baseline comune: finisce 10 ns prima del gate piu' anticipato ---
    bl_comune = (BASELINE_NS[0], min(BASELINE_NS[1], gate_intero[0] - 10, gate_s[0] - 10))
    if np.count_nonzero((t >= bl_comune[0]) & (t <= bl_comune[1])) < 10:
        raise ValueError(f"{DESCRIZIONE[lab]}: baseline comune {bl_comune} ns troppo corta")
    media_bl = media - media[(t >= bl_comune[0]) & (t <= bl_comune[1])].mean()
    integrale = lambda g: media_bl[(t >= g[0]) & (t <= g[1])].sum()

    g_stretto = q_pe_gate_stretto(compute_charge(wf_l, gate_s, bl_comune), lab)

    # --- Scan: inizio fisso a gate_s[0], fine da gate_s[0] + passo al gate intero ---
    fini = np.arange(gate_s[0] + PASSO_SCAN_NS, gate_intero[1] + 1e-9, PASSO_SCAN_NS)
    gates = [(gate_s[0], fine) for fine in fini]
    risultati = {}
    for g in gates + [gate_s, gate_intero]:
        if g in risultati:
            continue
        q_g = compute_charge(wf_l, g, bl_comune)
        c, passo, x0 = contrasto_picchi(q_g, g_stretto * integrale(g) / integrale(gate_s))
        risultati[g] = {"G": passo, "x0": x0, "contrasto": c, "q": q_g}
    confronto_gate[lab] = {"gate_intero": gate_intero, "baseline": bl_comune, "risultati": risultati}

    larg = np.array([g[1] - g[0] for g in gates])
    contr = np.array([risultati[g]["contrasto"] for g in gates])
    i_best = int(np.argmax(contr))
    r_s, r_i = risultati[gate_s], risultati[gate_intero]
    print(f"=== {DESCRIZIONE[lab]} ===  baseline comune ({bl_comune[0]:.0f}, {bl_comune[1]:.0f}) ns")
    print(f"  gate stretto {gate_s} ns:  G = {r_s['G']:.0f} ADC, contrasto = {r_s['contrasto']:.3f}")
    print(f"  gate intero  ({gate_intero[0]:.0f}, {gate_intero[1]:.0f}) ns:  G = {r_i['G']:.0f} ADC, "
          f"contrasto = {r_i['contrasto']:.3f}  ({r_s['contrasto'] / r_i['contrasto']:.0f} volte meno)")
    print(f"  scan: massimo {contr[i_best]:.3f} con gate ({gate_s[0]}, {gate_s[0] + larg[i_best]:.0f}) ns; "
          f"il gate stretto ne vale il {100 * r_s['contrasto'] / contr[i_best]:.0f}%")

    if not mostra(lab):
        continue
    fig = plt.figure(figsize=(20, 10))
    # Waveform media con i gate e la baseline comune
    ax0 = fig.add_subplot(2, 1, 1)
    ax0.plot(t, media, color="k", label="mean waveform")
    ax0.axvspan(*bl_comune, color="green", alpha=0.15,
                label=f"common baseline ({bl_comune[0]:.0f}, {bl_comune[1]:.0f}) ns")
    ax0.axvspan(*gate_intero, color="blue", alpha=0.10,
                label=f"full gate ({gate_intero[0]:.0f}, {gate_intero[1]:.0f}) ns")
    ax0.axvspan(*gate_s, color="orange", alpha=0.45, label=f"narrow gate {gate_s} ns")
    ax0.set_xlabel("Time [ns]"); ax0.set_ylabel("ADC"); ax0.grid(True, alpha=0.3); ax0.legend(fontsize=8)

    # Stessi eventi, due gate, in unita' di p.e.: (carica - piedistallo) / G
    ax1 = fig.add_subplot(2, 3, 4)
    bins_pe = np.arange(-3, 9, 0.05)
    for g, nome, col in [(gate_s, "narrow gate", "darkorange"), (gate_intero, "full gate", "blue")]:
        r = risultati[g]
        dens, _ = np.histogram((r["q"] - r["x0"]) / r["G"], bins=bins_pe, density=True)
        ax1.step(0.5 * (bins_pe[1:] + bins_pe[:-1]), dens, where="mid", color=col,
                 label=f"{nome}: contrast = {r['contrasto']:.3f}")
    for n in range(9):
        ax1.axvline(n, color="gray", lw=0.5, ls=":")
    ax1.set_xlabel("Charge [p.e.]"); ax1.set_ylabel("Probability density")
    ax1.set_title("Same events, two gates")
    ax1.grid(True, alpha=0.3); ax1.legend(fontsize=8)

    # Istogramma ripiegato: carica modulo G, piedistallo a 0. Piatto (= 1) se
    # non c'e' periodicita'; il contrasto e' l'ampiezza della sua prima armonica.
    ax_r = fig.add_subplot(2, 3, 5)
    bins_r = np.linspace(-0.5, 0.5, 41)
    for g, nome, col in [(gate_s, "narrow gate", "darkorange"), (gate_intero, "full gate", "blue")]:
        r = risultati[g]
        fase = ((r["q"] - r["x0"]) / r["G"] + 0.5) % 1 - 0.5
        dens, _ = np.histogram(fase, bins=bins_r, density=True)
        ax_r.step(0.5 * (bins_r[1:] + bins_r[:-1]), dens, where="mid", color=col,
                  label=f"{nome}: contrast = {r['contrasto']:.3f}")
    ax_r.axhline(1, color="gray", ls=":", label="no periodicity")
    ax_r.set_xlim(-0.5, 0.5); ax_r.set_ylim(0, None)
    ax_r.set_xlabel("Charge modulo 1 p.e. [p.e.]"); ax_r.set_ylabel("Probability density")
    ax_r.set_title("Charge folded every 1 p.e.")
    ax_r.grid(True, alpha=0.3); ax_r.legend(fontsize=8)

    # Scan del contrasto contro la larghezza del gate
    ax2 = fig.add_subplot(2, 3, 6)
    ax2.plot(larg, contr, "o-", color="k", ms=3, label=f"gate starting at {gate_s[0]} ns")
    ax2.plot(gate_s[1] - gate_s[0], r_s["contrasto"], "o", color="darkorange", ms=10,
             label=f"narrow gate {gate_s} ns")
    ax2.plot(gate_intero[1] - gate_intero[0], r_i["contrasto"], "s", color="blue", ms=9,
             label=f"full gate ({gate_intero[0]:.0f}, {gate_intero[1]:.0f}) ns")
    # sotto questo livello il contrasto e' quello di eventi senza nessuna periodicita'
    ax2.axhline(1 / np.sqrt(len(wf_l)), color="gray", ls=":", label="statistical floor (1/$\\sqrt{N}$)")
    ax2.set_ylim(0, None)
    ax2.set_xlabel("Gate width [ns]"); ax2.set_ylabel("Peak contrast")
    ax2.set_title("Peak visibility vs gate width")
    ax2.grid(True, alpha=0.3); ax2.legend(fontsize=8)

    fig.suptitle(f"{DESCRIZIONE[lab]} — why a narrow gate ({'filtered' if USA_FILTRATE else 'raw'} waveforms)")
    plt.tight_layout()
    plt.show()


In [ ]:
# === Waveform che formano ciascun picco della distribuzione di carica ===
# Le finestre dei picchi vengono dal fit di Poisson generalizzata (fit_gain2),
# non dall'istogramma: ogni picco n p.e. e' la gaussiana disegnata nella cella
# del fit (peso, posizione e larghezza dal fit). La finestra del picco n e'
# dove la sua gaussiana e' la piu' alta di tutte: li' un evento ha piu'
# probabilita' di essere n p.e. che qualsiasi altra cosa, ed e' la divisione
# che sbaglia meno. Un picco che non e' mai il piu' alto (a 52 V succede ai
# picchi coperti dal piedistallo) non ha finestra e non si mostra.
#
# Dall'area delle gaussiane dentro ogni finestra:
#  - purezza:    frazione degli eventi nella finestra che sono davvero n p.e.
#  - efficienza: frazione degli eventi n p.e. che cadono nella finestra
# Con picchi sovrapposti la purezza scende, e la waveform media della finestra
# mescola picchi diversi: va letta insieme alla purezza.
# Si usa la carica del gate stretto (charge_gain2), quella del fit.
# Solo visualizzazione: non cambia nulla delle celle precedenti.
from scipy.special import erf

# Si mostrano tutti i picchi con almeno MIN_EVENTI_PICCO eventi attesi: e' la
# statistica, non un numero fisso, a decidere fin dove arrivare (a 56.5 V
# oltre 10 p.e., a 52 V 3). N_PICCHI_MAX serve solo a tagliare i grafici
# quando sono troppi; None = nessun taglio.
N_PICCHI_MAX = None           # picchi mostrati per run, al massimo; None = tutti
MIN_EVENTI_PICCO = 100        # picchi con meno eventi attesi nella finestra non si mostrano
MAX_WF_PER_PICCO = 1000       # linee disegnate per picco (a caso); None = tutte (lento)
SEED_PICCHI = 0


def finestre_dal_fit(r, x_min, x_max):
    """[(n, inizio, fine)] delle regioni in cui la gaussiana del picco n e' la piu' alta."""
    mu, lam, s0, s1 = r["popt"][:4]
    m_tutti = posizioni_picchi(r["posizioni"], r["n_tot"])
    xx = np.linspace(x_min, x_max, 20000)
    comp = np.array([picco_singolo(xx, n, mu, lam, s0, s1, m_tutti[n]) for n in range(r["n_tot"])])
    vince = np.argmax(comp, axis=0)
    cambi = np.flatnonzero(np.diff(vince)) + 1
    inizi = np.r_[0, cambi]
    fini = np.r_[cambi, len(xx)]
    finestre = []
    for i0, i1 in zip(inizi, fini):
        a = -np.inf if i0 == 0 else 0.5 * (xx[i0 - 1] + xx[i0])
        b = np.inf if i1 == len(xx) else 0.5 * (xx[i1 - 1] + xx[i1])
        finestre.append((int(vince[i0]), a, b))
    return finestre


def frazione_gaussiana(a, b, m, s):
    """Frazione di una gaussiana (m, s) dentro [a, b)."""
    cdf = lambda x: 0.5 * (1 + erf((x - m) / (s * np.sqrt(2))))
    return cdf(b) - cdf(a)


finestre_gain2 = {}
for lab_p in fit_gain2:
    r = fit_gain2[lab_p]
    q_p = charge_gain2[lab_p]
    wf_p = wf_analisi[lab_p]
    h = hist_gain2[lab_p]
    mu, lam, s0, s1 = r["popt"][:4]
    m_tutti = posizioni_picchi(r["posizioni"], r["n_tot"])
    sig = lambda n: np.sqrt(s0 ** 2 + n * s1 ** 2)
    n_eventi = len(q_p)

    finestre = []
    for n, a, b in finestre_dal_fit(r, q_p.min(), q_p.max()):
        # eventi attesi nella finestra, picco per picco: N * P(k) * frazione della gaussiana k dentro
        attesi = np.array([n_eventi * gen_poisson_pmf(k, mu, lam) * frazione_gaussiana(a, b, m_tutti[k], sig(k))
                           for k in range(r["n_tot"])])
        osservati = np.count_nonzero((q_p >= a) & (q_p < b))
        # finestre lontane da tutto (coda estrema) hanno area attesa nulla: niente purezza
        if attesi.sum() <= 0:
            continue
        finestre.append({"n": n, "a": a, "b": b, "osservati": osservati, "attesi_n": attesi[n],
                         "purezza": attesi[n] / attesi.sum(),
                         "efficienza": frazione_gaussiana(a, b, m_tutti[n], sig(n))})
    finestre = [f for f in finestre if f["attesi_n"] >= MIN_EVENTI_PICCO][:N_PICCHI_MAX]
    if not finestre:
        print(f"=== {DESCRIZIONE[lab_p]} === nessun picco con almeno {MIN_EVENTI_PICCO} eventi attesi")
        continue
    finestre_gain2[lab_p] = finestre

    print(f"=== {DESCRIZIONE[lab_p]} ===" + ("" if r["affidabile"] else "  (fit non affidabile)"))
    print(f"{'picco':>7} {'finestra [ADC]':>22} {'eventi':>8} {'attesi n p.e.':>14} {'purezza':>8} {'efficienza':>11}")
    for f in finestre:
        print(f"{f['n']:>5} pe [{f['a']:9.0f}, {f['b']:9.0f}) {f['osservati']:8d} {f['attesi_n']:14.0f} "
              f"{100 * f['purezza']:7.1f}% {100 * f['efficienza']:10.1f}%")

    if not mostra(lab_p):
        continue
    col_pe = lambda n: plt.cm.tab10([1, 2, 4, 5, 6, 8, 9][n % 7])   # come nella cella del fit

    # --- Panoramica: istogramma, fit, gaussiane e finestre ---
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.step(h["centers"], h["counts"], where="mid", color="tab:blue", label=DESCRIZIONE[lab_p])
    xx = np.linspace(h["centers"][0], h["centers"][-1], 2000)
    norm = h["counts"].sum() * h["bin_width"]
    ax.plot(xx, sum(picco_singolo(xx, k, mu, lam, s0, s1, m_tutti[k]) for k in range(r["n_tot"])) * norm,
            color="red", label=f"generalized Poisson fit, $\\chi^2$/ndf = {r['chi2_rid']:.2f}")
    for f in finestre:
        n = f["n"]
        ax.plot(xx, picco_singolo(xx, n, mu, lam, s0, s1, m_tutti[n]) * norm, ls="--", color=col_pe(n))
        a = max(f["a"], h["centers"][0]); b = min(f["b"], h["centers"][-1])
        ax.axvspan(a, b, color=col_pe(n), alpha=0.15,
                   label=f"{n} p.e.: {f['osservati']} events, purity {100 * f['purezza']:.0f}%, "
                         f"efficiency {100 * f['efficienza']:.0f}%")
    ax.set_xlabel("Charge [ADC]"); ax.set_ylabel(f"Events / {h['bin_width']:.1f} ADC")
    ax.set_title(f"{DESCRIZIONE[lab_p]} — peak windows from the fit (gate {GATE_NS_RUN[lab_p][0]:.0f}-{GATE_NS_RUN[lab_p][1]:.0f} ns)")
    ax.grid(True, alpha=0.3)
    # legenda fuori: con 10-15 picchi dentro coprirebbe la distribuzione
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    plt.tight_layout()
    plt.show()

    # --- Un grafico per picco: waveform e loro media, stessa scala y per tutti i picchi del run ---
    rng_p = np.random.default_rng(SEED_PICCHI)
    y_lo, y_hi = np.percentile(wf_p[:2000], [0.1, 99.9])
    medie = {}
    for f in finestre:
        n = f["n"]
        sel = np.flatnonzero((q_p >= f["a"]) & (q_p < f["b"]))
        if len(sel) == 0:
            continue
        medie[n] = wf_p[sel].mean(axis=0)
        disegna = sel if MAX_WF_PER_PICCO is None or len(sel) <= MAX_WF_PER_PICCO \
            else rng_p.choice(sel, MAX_WF_PER_PICCO, replace=False)
        # fig, ax = plt.subplots(figsize=(15, 4.5))
        # ax.plot(t, wf_p[disegna].T, color=col_pe(n), lw=0.3, alpha=0.08, rasterized=True)
        # ax.plot(t, medie[n], color="k", lw=2, label=f"mean over all {len(sel)} events")
        # ax.axvspan(*BASELINE_NS, color="green", alpha=0.12, label=f"baseline {BASELINE_NS} ns")
        # ax.axvspan(*GATE_NS_RUN[lab_p], color="orange", alpha=0.25, label=f"gate {GATE_NS_RUN[lab_p]} ns")
        # ax.set_ylim(y_lo, y_hi)
        # ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
        # ax.set_title(f"{DESCRIZIONE[lab_p]} — {n} p.e. (purity {100 * f['purezza']:.0f}%): "
        #              f"{len(disegna)} of {len(sel)} waveforms drawn")
        # plt.tight_layout()
        # plt.show()

    # --- Le medie dei picchi sovrapposte: l'ampiezza deve crescere di un passo per p.e. ---
    if medie:
        fig, ax = plt.subplots(figsize=(15, 4.5))
        # scala di colore ordinata: i 7 colori di col_pe si ripeterebbero oltre 6 p.e.
        n_max = max(medie)
        for n, med in medie.items():
            ax.plot(t, med, color=plt.cm.viridis(n / max(n_max, 1)), lw=1.5, label=f"{n} p.e.")
        ax.axvspan(*GATE_NS_RUN[lab_p], color="orange", alpha=0.15, label=f"gate {GATE_NS_RUN[lab_p]} ns")
        ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
        ax.set_title(f"{DESCRIZIONE[lab_p]} — mean waveform of each peak")
        plt.tight_layout()
        plt.show()


In [ ]:
# === Scelta della tensione di lavoro: cosa si guadagna e cosa si paga ===
# Tutto dai run col LED, celle precedenti: Q_pe_tot e guadagno (breakdown),
# lambda (fit di Poisson generalizzata), PDE dal datasheet con V_bd misurato.
#
# Cosa entra nella scelta, per un veto a scintillatore:
#  - PDE: i fotoelettroni PRIMARI per muone le sono proporzionali, e da loro
#    dipende l'efficienza. Si mostra la curva del datasheet a 450 nm (picco di
#    sensibilita', la piu' vicina a uno scintillatore plastico blu, ~420-430 nm)
#    e quella a 400 nm usata per il LED. Se la luce passa da fibre WLS (~490 nm)
#    va usata la curva a quella lunghezza d'onda.
#  - crosstalk lambda: un impulso al buio diventa 2, 3... p.e. con probabilita'
#    P(>=k) della Borel. Il rate di falsi sopra k p.e. e' DCR * P(>=k): qui si
#    da' P(>=k), il DCR si misura sui pannelli col trigger esterno.
#    Il crosstalk allarga anche la distribuzione di carica: per un primario la
#    Borel ha varianza/media^2 = lambda/(1-lambda), cioe' un fattore di rumore
#    in eccesso ENF = 1/(1-lambda) sulla varianza.
#  - guadagno: qui e' gia' abbondante a tutte le tensioni (impulso da 1 p.e.
#    ben sopra il rumore da 53.5 V); non e' il parametro che decide.
#
# Incertezza su V_bd: l'errore statistico del fit (~0.02 V) non e' quella vera.
# Rifatto solo sulle tensioni da 53.5 V in su, V_bd esce 51.51 invece di 51.43:
# si usa +-0.1 V di sistematico, che sposta la PDE come mostrato.
from math import factorial

E_VBD_SIST = 0.1     # [V] sistematico su V_bd (dipendenza dall'intervallo di tensioni del fit)


def borel_coda(k, lam):
    """P(>= k celle) per un fotoelettrone primario (Borel)."""
    return 1 - sum(np.exp(-n * lam) * (n * lam) ** (n - 1) / factorial(n) for n in range(1, k))


def pde_450(dv):
    """PDE [frazione] a 450 nm (curva del datasheet) per sovratensione dv [V]."""
    return modello_pde(dv, *p_pde) / 100


scelta = {}
for l in fit_gain2:
    r = fit_gain2[l]
    if not r["affidabile"]:
        continue
    dv = VBIAS[l] - V_BD
    if dv <= 0:
        continue
    lam = r["lam"]
    scelta[l] = {
        "dv": dv, "G": r["q_pe_tot"] * ELETTRONI_PER_ADC, "e_G": r["e_q_pe_tot"] * ELETTRONI_PER_ADC,
        "pde450": pde_450(dv), "pde450_lo": pde_450(max(dv - E_VBD_SIST, 1e-3)), "pde450_hi": pde_450(dv + E_VBD_SIST),
        "pde400": pde_led(dv), "lam": lam, "e_lam": r.get("e_lam", np.nan), "enf": 1 / (1 - lam),
        "p2": borel_coda(2, lam), "p3": borel_coda(3, lam), "p4": borel_coda(4, lam),
        "estrapolata": dv < PDE_DV_V[0],
        # nei run tenuti per FORZA_AFFIDABILE i picchi non si separano: Q_pe si', lambda no
        "lam_misurato": l not in FORZA_AFFIDABILE,
    }

print(f"V_bd = {V_BD:.2f} V (± {E_VBD_SIST} V sistematico).  PDE: curva del datasheet; "
      f"sotto dV = {PDE_DV_V[0]:g} V e' estrapolata (*).")
print(f"{'V':>6} {'dV':>5} {'G [1e6]':>9} {'PDE 450 nm':>18} {'PDE 400':>8} {'lambda':>15} {'ENF':>6} "
      f"{'P(>=2)':>7} {'P(>=3)':>7} {'P(>=4)':>8}")
for l, s in scelta.items():
    print(f"{VBIAS[l]:6.1f} {s['dv']:5.2f} {s['G'] / 1e6:5.2f}±{s['e_G'] / 1e6:4.2f} "
          f"{100 * s['pde450']:5.1f}% ({100 * s['pde450_lo']:4.1f}-{100 * s['pde450_hi']:4.1f}) "
          f"{100 * s['pde400']:7.1f}% {s['lam']:6.3f} ± {s['e_lam']:5.3f} {s['enf']:6.2f} "
          f"{100 * s['p2']:6.1f}% {100 * s['p3']:6.2f}% {100 * s['p4']:7.3f}%" + (" *" if s["estrapolata"] else "")
          + ("" if s["lam_misurato"] else "  lambda NON misurato: picchi non separati"))

# --- Il compromesso in un grafico: cosa si guadagna (PDE) e cosa si paga (crosstalk) ---
ls = list(scelta)
x = np.array([VBIAS[l] for l in ls])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
p450 = np.array([scelta[l]["pde450"] for l in ls]) * 100
ax1.fill_between(x, [100 * scelta[l]["pde450_lo"] for l in ls], [100 * scelta[l]["pde450_hi"] for l in ls],
                 color="tab:blue", alpha=0.2, label=f"PDE 450 nm, $V_{{bd}}$ ± {E_VBD_SIST} V")
ax1.plot(x, p450, "o-", color="tab:blue", label="PDE 450 nm (datasheet curve)")
ax1.plot(x, [100 * scelta[l]["pde400"] for l in ls], "s--", color="tab:purple", mfc="white", label="PDE 400 nm (LED)")
ax1.set_xlabel("Bias voltage [V]"); ax1.set_ylabel("PDE [%]")
ax1.set_title("What you gain: photon detection efficiency")
ax1.set_ylim(0, None); ax1.grid(True, alpha=0.3); ax1.legend(fontsize=8)
sec = ax1.secondary_xaxis("top", functions=(lambda v: v - V_BD, lambda d: d + V_BD))
sec.set_xlabel("Overvoltage [V]")
lm = [l for l in ls if scelta[l]["lam_misurato"]]
for k, mk in ((2, "o"), (3, "s"), (4, "^")):
    ax2.plot([VBIAS[l] for l in lm], [100 * scelta[l][f"p{k}"] for l in lm], marker=mk, label=f"one dark pulse looks like $\\geq${k} p.e.")
ax2.set_yscale("log")
ax2.set_xlabel("Bias voltage [V]"); ax2.set_ylabel("Probability [%]")
ax2.set_title("What you pay: crosstalk lifts dark pulses above the threshold")
ax2.grid(True, which="both", alpha=0.3); ax2.legend(fontsize=8)
sec2 = ax2.secondary_xaxis("top", functions=(lambda v: v - V_BD, lambda d: d + V_BD))
sec2.set_xlabel("Overvoltage [V]")
plt.tight_layout()
plt.show()


In [ ]:
# === Risultati in measurements/: quello che serve al light yield dei pannelli ===
# LY [p.e.] = Q del picco dei muoni / Q_pe_tot, con la carica dei muoni
# integrata COME R: stessa catena (canale, scheda, filtro), baseline per evento
# presa prima dell'impulso come media dei campioni, finestra lunga che comincia
# e finisce alla stessa distanza dalla salita (qui dichiarata rispetto
# all'istante in cui l'impulso medio supera l'80% del massimo, cioe' l'inizio
# del gate). Q_pe_tot conta le celle accese, crosstalk compreso: i
# fotoelettroni primari sono circa Q / Q_pe_tot * (1 - lambda).
# Q_pe_tot dipende dalla temperatura (V_bd ~54 mV/°C: a dV ~3 V, ~1.8%/°C):
# se i cosmici si prendono in un altro periodo, ricontrollare Q_pe sui loro
# impulsi al buio.
# Si scrivono anche gli istogrammi del fit, per rifare i grafici senza i file grezzi.
import json, datetime

DIR_MISURE = os.path.abspath(os.path.join("..", "measurements"))
data_run = os.path.basename(os.path.dirname(DATA_DIR))       # es. "10072026"
nome_json = os.path.join(DIR_MISURE, f"sipm_gain_led_{os.path.basename(DATA_DIR)}_{data_run}.json")

per_run = {}
for l in fit_gain2:
    r = fit_gain2[l]
    g = GATE_NS_RUN[l]
    per_run[l] = {
        "file": RUNS_G2[l], "vbias_V": VBIAS[l], "led_ampl": None if np.isnan(LED_AMPL[l]) else LED_AMPL[l],
        "gate_ns": list(g), "gate_da_run": gate_info[l]["da"],
        "q_pe_gate_adc": r["q_pe"], "e_q_pe_gate_adc": r["e_q_pe"],
        "R": r["R_gate"], "e_R": r["e_R_gate"], "R_misurato_qui": bool(r["R_affidabile"]),
        "q_pe_tot_adc": r["q_pe_tot"], "e_q_pe_tot_adc": r["e_q_pe_tot"],
        "guadagno": r["q_pe_tot"] * ELETTRONI_PER_ADC, "e_guadagno": r["e_q_pe_tot"] * ELETTRONI_PER_ADC,
        "lambda": r["lam"], "e_lambda": r["e_lam"], "mu": r["mu"], "e_mu": r["e_mu"],
        "chi2_ndf": r["chi2_rid"], "affidabile": bool(r["affidabile"]),
        "finestra_lunga_rispetto_salita80_ns": [FINESTRA_SEGNALE_NS[0] - g[0], FINESTRA_SEGNALE_NS[1] - g[0]],
        "istogramma": {"bordi_adc": hist_gain2[l]["bin_edges"].tolist(), "conteggi": hist_gain2[l]["counts"].tolist()},
    }

misura = {
    "cosa": "SiPM con LED: carica per fotoelettrone nel gate e su tutto l'impulso, crosstalk, breakdown",
    "scritto_il": datetime.datetime.now().isoformat(timespec="seconds"),
    "notebook": "tools/SiPM_Characterization_Gain_Study.ipynb",
    "dati": os.path.relpath(DATA_DIR, os.path.abspath("..")), "canale": CHANNEL,
    "sampling_time_ns": SAMPLING_TIME,
    "filtro": {"lowpass": FILTRO_LOWPASS, "lp_cutoff_mhz": LP_CUTOFF_MHZ, "lp_ordine": LP_ORDINE,
               "notch": FILTRO_NOTCH, "usa_filtrate": USA_FILTRATE},
    "baseline_ns": list(BASELINE_NS), "baseline": "media dei campioni, per evento",
    "gate": {"criterio": f"dal {100 * GATE_FRAZ_SALITA:.0f}% della salita della waveform media, largo {GATE_LARGHEZZA_NS} ns",
             "frazione_salita": GATE_FRAZ_SALITA, "larghezza_ns": GATE_LARGHEZZA_NS},
    "finestra_lunga_ns": list(FINESTRA_SEGNALE_NS),
    "finestra_lunga_criterio": f"dalla fine della baseline al {100 * FRAZ_CARICA_FINESTRA:.0f}% della carica cumulata, arrotondato a 50 ns",
    "catena": {"scheda_guadagno": GUADAGNO_AMPLI, "r_ingresso_ohm_assunta": R_INGRESSO_OHM,
               "elettroni_per_adc_campione": ELETTRONI_PER_ADC},
    "v_bd_V": V_BD, "e_v_bd_stat_V": E_V_BD, "e_v_bd_sist_V": E_VBD_SIST,
    "v_bd_nota": "il sistematico viene dalla dipendenza dall'intervallo di tensioni del fit (51.43 su 52-56.5 V, 51.51 da 53.5 V in su)",
    "pendenza_q_pe_adc_per_V": k_q, "pendenza_q_pe_tot_adc_per_V": k_t,
    "per_run": per_run,
}
with open(nome_json, "w") as fj:
    json.dump(misura, fj, indent=1, ensure_ascii=False)
print(f"Scritto {os.path.relpath(nome_json, os.path.abspath('..'))}: {len(per_run)} run")
print(f"{'V':>6} {'gate [ns]':>13} {'Q_pe gate':>14} {'R':>14} {'Q_pe_tot':>15} {'lambda':>14}")
for l, p in per_run.items():
    print(f"{p['vbias_V']:6.1f} {p['gate_ns'][0]:6.1f}-{p['gate_ns'][1]:5.1f} {p['q_pe_gate_adc']:7.1f} ± {p['e_q_pe_gate_adc']:4.1f} "
          f"{p['R']:6.3f} ± {p['e_R']:5.3f} {p['q_pe_tot_adc']:8.1f} ± {p['e_q_pe_tot_adc']:4.1f} "
          f"{p['lambda']:6.3f} ± {p['e_lambda']:5.3f}" + ("" if p["affidabile"] else "  non affidabile"))
